**Change log · Chest pain NMF + PrefixSpan (fused)**

- **v1-v3** `Day1_nmf_v*`, `Day3_prefixspan_v*`  
  Separate Day 1 / Day 3 notebooks (untouched).
- **full** `Chestpain_NMF_PrefixSpan_full` (2026-10-09)  
  Day 1 (NMF) and Day 3 (PrefixSpan) fused as Part A and B; all events of each patient, no window.
- **AHA** `Chestpain_NMF_PrefixSpan_AHA` (2026-10-14)  
  `..._full` on `codesets_v2.yaml` (2021 AHA/ACC guideline, Gulati et al., Circulation 2021;144:e368); `DESCR` emptied.
- **agesex** `Chestpain_NMF_PrefixSpan_AHA_agesex` (2026-10-27, revised 10-28), after the 2026-10-26 review (age, gender, nitrates)  
  `codesets_v3.yaml`, sparse bag, `AGE_SEX_CROSS` modes, Part A+ / A++, ladder; four copies in `age_sex_sensitivity/`.
- **v2** `Chestpain_NMF_PrefixSpan_AHA_v2` (2026-10-29)  
  `'extended'` as the default, plus the "where the sex and age columns sit" cells. Saved run: k = 8, ARI 0.825 vs `..._AHA`.
- **sean** `Chestpain_NMF_PrefixSpan_sean` (2026-11-02) ← **this notebook**  
  `codesets_sean.yaml` (Sean's underwriting view, standalone); new mode `'explicit'` = plain codes + two 0/1 flags, man aged 45+ and woman aged 55+ at a chest pain visit; `'extended'` unchanged; Part A++ removed.

*Open: `DESCR` is written for the k = 8 fit (all eight components solid); Part B and the new cells have not been run. The narrative cells (readings of k, "c3 is ...", tag names such as `panic_attack`, `ssri`, `stress_test`) describe the `..._AHA` k=8 run on `codesets_v3.yaml`.*

# Chest pain · NMF bundles and sequence motifs — Sean codesets with explicit high-risk age and sex flags

**Part A (Day 1)**: events, bag of codes, NMF (k from the stability sweep), loadings, NMF-bundles table, components JSON.  
**Part B (Day 3)**: PrefixSpan motifs per bundle, stability, comparison groups, trajectory graph, final table.

Set `VARIANT`, `WINDOW_MONTHS`, `CODESETS_FILE`, `TOKEN_LEVEL` and (optionally) `K_OVERRIDE` in the next cell, plus the new `SMOKE`, `AGE_SEX_CROSS`, its five values (`'plain'`, `'crossed'`, `'both'`, `'extended'`, `'explicit'`), the weights `W_S`, `W_D` and `W_R`, `RISK_RULES` and `AGE_EDGES`.  
`TOKEN_LEVEL='concept'` reproduces the `k5_full` NMF (same tokens, only anchors / labels / coverage come from v2); `'group'` makes each codeset tag the NMF token, which is the real test of v2.

**This notebook runs on `codesets_sean.yaml`** (Sean's underwriting view, standalone: no v3 tags) **with the new mode `AGE_SEX_CROSS = 'explicit'`**, the default: the plain codes plus two 0/1 columns, man aged 45+ and woman aged 55+ at any chest pain visit (`RISK_RULES`, weight `W_R`), not crossed with any code. `'extended'` keeps its meaning (sex and age band one-hots). The other modes (`'plain'`, `'crossed'`, `'both'`) run by changing that line; their executed copies sit in `age_sex_sensitivity/`. Every file it writes carries the mode in its name (`..._sean_<mode>`). Paths: inputs and sql two folders up; helpers, codesets and the AHA k=8 fit in this folder.

In [ ]:
AGE_SEX_CROSS = 'explicit'  # what the bag of codes holds: 'plain' = only the plain codes (the AHA k=8 matrix) | 'crossed' = only crossed features, every code x sex x age band AT THE EVENT ('anxiety_F_35-44', 'blood_cell_count_aut__procedure__4035541_F_35-44') | 'both' = plain + crossed | 'extended' = plain + the patient's sex and age band AT EACH chest pain event (every recurrent event lights its own band), one-hot, no crossing (weight W_D) | 'explicit' = plain + two one-hot flags of Sean's high-risk rule, man aged 45+ and woman aged 55+, met at ANY chest pain visit (RISK_RULES, weight W_R), no crossing
assert AGE_SEX_CROSS in ('plain', 'crossed', 'both', 'extended', 'explicit'), AGE_SEX_CROSS
VARIANT       = f'sean_{AGE_SEX_CROSS}'   # label in TAG, the npz, the components json and both xlsx; it carries the AGE_SEX_CROSS mode so the 'plain', 'crossed' and 'both' runs never overwrite each other ('AHA' was the k=8 run, 'full' the k5_full run). Windowing depends only on WINDOW_MONTHS
WINDOW_MONTHS = None        # applied right after tagging (ethos_omop.window_around_anchor); None = every event of each person
K_OVERRIDE    = None       # None = the k chosen by the stability sweep; set an int (e.g. 7) to force it
CODESETS_FILE = 'codesets_sean.yaml'   # Sean's underwriting view, standalone (no v3 tags); 'codesets_v3.yaml' = the AHA v3 file of the v2 notebook
TOKEN_LEVEL   = 'group'     # 'group': a codeset tag is the NMF token (curated concepts collapse) | 'concept': one token per concept (as k5_full)
ANCHOR        = 'chest_pain'
COHORT        = 'chestpain'
TAG           = f"{COHORT}_{VARIANT}"      # used in every cache / json / xlsx file name
assert TOKEN_LEVEL in ('group', 'concept'), TOKEN_LEVEL

# ---- new in this notebook: age at exposure and sex
SMOKE         = False      # True: 5,000 random patients, a short k grid and few refits, to see the whole notebook run in minutes (Part B has little to mine on so few patients)
SMOKE_N       = 5_000
PLAIN = AGE_SEX_CROSS in ('plain', 'both', 'extended', 'explicit')      # the bag has the plain columns
CROSS = AGE_SEX_CROSS in ('crossed', 'both')                # the bag has the crossed columns (one extra pass over the events)
DEMO  = AGE_SEX_CROSS in ('extended', 'explicit')            # the matrix has demographic columns, never crossed with a code: 'extended' the sex and age band one-hots (block D), 'explicit' the two risk flags (block R)
FLAGB = {'extended': 'D', 'explicit': 'R'}.get(AGE_SEX_CROSS)    # the block of those columns (None for the other modes)
W_S           = 1.0        # value of the 0/1 crossed columns when plain + crossed are fitted together ('both', and the ladder's 'plain + S' row); a scale does not change a crossed-only fit
W_D           = 0.5        # value of the 0/1 sex and age band columns in 'extended' (and the ladder's 'plain + D' row). At 1.0 they are among the commonest columns and tend to take components of their own (first round: sex=F, sex=M); lower it to let them join the care bundles
AGE_EDGES     = (25, 35, 45, 55, 65)   # 10-year bands: '<25', '25-34', '35-44', '45-54', '55-64', '65+' (the cohort is 99% 25-64 at the first chest pain, check the tails in 5b). (25, 30, ..., 65) gives the Census 5-year bands, tuple(range(0, 90, 5)) the plain ones
W_R           = 0.5        # value of the two 0/1 risk flags in 'explicit' (and the ladder's 'plain + R' row); same reasoning as W_D
RISK_RULES    = (('male_45plus', 'M', 45), ('female_55plus', 'F', 55))   # Sean's high-risk age/sex rule: (column 'demo_<name>', sex, minimum age); the column is 1 if ANY chest pain visit is on or after that birthday
if SMOKE and K_OVERRIDE is None:
    K_OVERRIDE = 8          # a 5,000 patient sweep is too noisy to pick k


# Part A · Day 1 — NMF bundles

## 0 · Getting the data

`ethos_omop.extract_cohort()` pulls the dx_9487 cohort: any patient carrying one of the
five R07.x codes, defined in `sql/chest_pain_cohort.sql`. It dumps the same shape
`extract_omop()` does — `master_*` as the source of truth, `ethos_dip` excluded on both
`concept_id` and `source_concept_id` for every domain table, standard-named symlinks for
`meds_etl_omop`, plus:

* **`concept.csv` slimmed** to the concept_ids the cohort actually references, instead of
  the full multi-million-row table.
* **`concept_ancestor` dumped too**, restricted to rows touching the cohort — needed for
  ontology rollup (unspecified chest-pain codes to their parents, drugs to ingredients).

```python
import ethos_omop as eo
eo.extract_cohort("../sql/chest_pain_cohort.sql")   # -> ../data_input/
```

The next cell runs this once (150,368 patients, all real Ethos production data) and skips
it on every re-run once `../data_input/` exists.

Without Snowflake, or when you want data whose answer you already know:

```bash
python make_demo_extract.py 2000
```

The demo extract has the **same schema** — integer `concept_id`s resolved through
`concept.csv`, `master_*` table names, the symlinks — plus a `truth.csv` giving each
patient's planted trajectory archetype. Real extracts have no `truth.csv`. That absence is
the whole problem this course is about, and every notebook here works either way.

*In plain terms: this is a one-time data export, like running a big report and saving the results as spreadsheets (CSV files) in a folder. Every cell below reads from that folder, not from Snowflake directly.*

In [ ]:
import os, sys
import ethos_omop as eo

OUT_DIR     = "../../inputs/chest_pain_cohort/"                     # extract_cohort() output, relative to this notebook's folder
COHORT_SQL  = "../../sql/chest_pain_cohort.sql"   # relative to this notebook's cwd

# Runs once: if OUT_DIR is already populated, notebooks load from disk and never
# re-hit Snowflake. Delete OUT_DIR (or pick a different one) to force a re-pull.
if not os.path.isdir(OUT_DIR):
    eo.extract_cohort(COHORT_SQL, out_dir=OUT_DIR)
else:
    print(f"{OUT_DIR}/ already populated -- skipping extract_cohort(). "
          f"Delete it to re-pull from Snowflake.")

### v2 · Visit pruning (care-driven bias)

The bundles from v1 reflect **care delivery, not condition**: pharmacy, office and ED visits are the
most common tokens and pull unrelated patients together. v2 changes one thing, at the load step
below: every event whose `domain` or `vocabulary_id` is "visit" is dropped before tagging.
Everything after it is unchanged, so v1 and v2 results are directly comparable.

Side effects to expect: the `ed_visit` codeset tag disappears (it sits on visit concepts), the
Table 1 `ed_visit` row reads 0%, and the proxy lookback window can shrink for patients whose first or
last record was a visit. The NMF fit is saved as `nmf_k{k}_chestpain_v2.npz`, so v1's file is not overwritten.


In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
pd.set_option('display.width', 150); pd.set_option('display.max_columns', 40)

DOMAINS = ['Visit', 'Condition', 'Drug', 'Procedure', 'Measurement']   # Visit stays in the load so we can show before vs after

concept  = eo.load_concept(OUT_DIR)
eo.CODESETS = eo._load_codesets(CODESETS_FILE)   # AHA run: eo.CODESETS is read at call time by build_codesets and by cells 13 and 41
eo.PSYCH_TAGS   = ['anxiety']                                        # chest_pain_labels asserts that every tag of these two lists exists in the code sets; the Sean file has no panic / SSRI / benzodiazepine tags
eo.CARDIAC_TAGS = ['uw_cad_dx', 'uw_cardiac_testing', 'nitrates']     # probe labels only, never features; troponin is Sean's first-line / low-risk test, so it is not cardiac evidence here
codesets = eo.build_codesets(concept, verbose=True)
raw      = eo.load_events(OUT_DIR, domains=DOMAINS)
person   = eo.load_person(OUT_DIR)
truth    = eo.load_truth(OUT_DIR)                   # None against real data

# v2: drop every visit event BEFORE tagging (care-driven bias). A visit row is any row whose
# domain OR vocabulary is "visit"; pharmacy / office / ED visits otherwise dominate the bundles.
is_visit = (raw['domain'].astype(str).str.lower().eq('visit')
            | raw['vocabulary_id'].astype(str).str.lower().eq('visit'))
visits   = raw[is_visit]
events   = eo.tag_events(raw[~is_visit].reset_index(drop=True), codesets, token_level=TOKEN_LEVEL, multi_group=True)

# ---- before / after pruning
tbl = (pd.DataFrame({'before': raw['domain'].value_counts(), 'after': events['domain'].value_counts()})
         .fillna(0).astype(int))
tbl.loc['TOTAL'] = [len(raw), len(events)]
tbl['removed_%'] = ((1 - tbl['after'] / tbl['before']) * 100).round(1)
print("events per domain, BEFORE vs AFTER visit pruning")
print(tbl.to_string())
print(f"\npatients:               {raw.person_id.nunique():>10,} -> {events.person_id.nunique():,}")
print(f"distinct concepts:      {raw.concept_id.nunique():>10,} -> {events.concept_id.nunique():,}")
print(f"median events/patient:  {raw.groupby('person_id').size().median():>10,.0f} -> "
      f"{events.groupby('person_id').size().median():,.0f}")
print("\nremoved rows by vocabulary:")
print(visits['vocabulary_id'].value_counts().head(10).to_string())
print("\ntop removed visit concepts:")
print(visits['concept_name'].value_counts().head(10).to_string())

visit_ids = set(visits['concept_id'])
lost      = sorted(t for t, ids in codesets.items() if set(ids) & visit_ids)
print("\ncurated codeset tags that sit on visit concepts (their events are gone):", lost or "none")
vanished  = set(raw['person_id']) - set(events['person_id'])
if vanished:
    print(f"WARNING: {len(vanished):,} patients had only visit events and are no longer in `events`")
del raw, visits                                      # free memory; `events` is the pruned stream from here on

print(f"\nevents {events.shape}   patients {events.person_id.nunique()}")
print(f"codeset coverage: {events.in_codeset.mean():.1%} of events carry a clinical tag")
print(f"truth labels: {'present (DEMO extract)' if truth is not None else 'ABSENT (real data)'}")
events.head(6)


In [ ]:
if WINDOW_MONTHS:
    _n0, _p0 = len(events), events.person_id.nunique()
    events = eo.window_around_anchor(events, ANCHOR, months=WINDOW_MONTHS)
    print(f"windowed to +/- {WINDOW_MONTHS} months around any {ANCHOR} event, merged per patient, exact duplicates dropped:")
    print(f"  events   {_n0:,} -> {len(events):,}")
    print(f"  patients {_p0:,} -> {events.person_id.nunique():,}")
    print(f"  median events per patient: {events.groupby('person_id').size().median():.0f}")
else:
    print(f"full variant: every event of each patient ({len(events):,} events, {events.person_id.nunique():,} patients)")


In [ ]:
if SMOKE:
    _keep  = np.random.default_rng(0).choice(events.person_id.unique(), SMOKE_N, replace=False)
    events = events[events.person_id.isin(_keep)].reset_index(drop=True)
    print(f"SMOKE: {events.person_id.nunique():,} patients, {len(events):,} events (set SMOKE = False for the full run)")

In [ ]:
print(eo.describe_extract(OUT_DIR).to_string(index=False))

## 1 · Four decisions 

Read these before trusting anything downstream. Each is a modelling choice, not a detail.

**1. Standard concepts only.** `load_events()` keeps `standard_concept == 'S'` and drops
`concept_id = 0` (unmapped). Note the exclusion idiom in the extract is
`IS DISTINCT FROM 'ethos_dip'`, not `<> 'ethos_dip'` — the latter silently drops NULLs.

*In plain terms: medical coding systems (ICD, SNOMED, etc.) have multiple ways to say the same thing. "Standard concepts" is OMOP's canonical, de-duplicated version of each code — we only keep those, so the same diagnosis isn't double-counted under two different code spellings.*

**2. `concept_code` stays a string.** Never let pandas parse it as numeric; leading zeros
and alphanumeric codes both break.

*In plain terms: some medical codes look like numbers (e.g. `"0042"`) but aren't meant to be treated as one — turning them into an actual number silently drops the leading zero and corrupts the code.*

**3. Claims carry a DATE, not a timestamp.** Same-day events have no true order.
`DOMAIN_ORDER` breaks ties deterministically — visit, then condition, procedure,
measurement, drug, observation. It is arbitrary and it is written down. Stanford's CLMBR
was built for minute-level resolution; we do not have that, and pretending otherwise is
where silent errors come from.

*In plain terms: claims data tells you "something happened on March 3rd," not "at 2:14pm." If five things happened the same day, we can't know their real order — so we pick one consistent order and are upfront that it's a guess, rather than pretending we have more precision than we do.*

**4. There is no `observation_period` in the extract.** `observation_window()` derives a
proxy from first and last observed event, which **understates** the true window. Anyone
whose history starts near their first event is left-truncated: "no prior diagnosis" may
mean "not observed yet."

*In plain terms: we only see what's in this data source. "This patient has no record of an anxiety diagnosis" might mean they truly don't have it, or it might just mean we started watching them too late to see it. Absence of a code is not proof of absence of the condition.*

In [ ]:
win = eo.observation_window(events)
print(win.lookback_days.describe().round(0).to_string())

plt.figure(figsize=(7, 3))
plt.hist(win.lookback_days / 365.25, bins=40, color='#2C5F6F')
plt.axvline(2, ls='--', c='firebrick', label='2-year minimum')
plt.xlabel('observed history (years)'); plt.ylabel('patients')
plt.title('proxy lookback -- left truncation is real'); plt.legend()
plt.tight_layout(); plt.show()

SHORT = win.loc[win.lookback_days < 730, 'person_id']
print(f"\n{len(SHORT)} patients ({len(SHORT)/len(win):.1%}) have under 2 years of history.")
print("Exclude them from any analysis that depends on the ABSENCE of a code.")

## 2 · Concept sets, resolved by name

`CODESETS` in `ethos_omop.py` maps a clinical label to name patterns plus vocabularies.
Resolving by name rather than hardcoding concept_ids means the same notebook runs against
real Snowflake concepts and the demo extract alike.

**Always eyeball the resolution before trusting it.** This is the step where a bad regex
quietly poisons everything downstream.

*In plain terms: instead of writing down a fixed list of magic numbers ("code 443698 means panic attack"), we write down a search pattern ("anything whose name matches panic_attack in the SNOMED vocabulary") and let the computer look up the matching codes every time. That way the same notebook keeps working even if the underlying code numbers differ between data sources.*

In [ ]:
# Eyeball it: what did the "anxiety" pattern actually match? Read
# straight from codesets.yaml (via eo.CODESETS) instead of retyping the
# patterns here, so this can't silently drift from what build_codesets()
# actually resolves.
pa_spec = eo.CODESETS['anxiety']
eo.resolve(concept, pa_spec['patterns'], vocabularies=pa_spec.get('vocabularies'))

In [ ]:
# On real data, roll a drug ingredient DOWN to all its products via concept_ancestor.
bz = codesets['nitrates']
print("nitrates concepts, direct:      ", len(bz))
print("nitrates concepts, + descendants:", len(eo.expand_descendants(bz)))

# And roll an unspecified diagnosis UP to its parents -- directly useful for the
# unspecified chest pain codes (R07.8, R07.89) this whole cohort is built from.
cp = codesets['chest_pain']
print("chest pain parents, 1 level up:  ", len(eo.roll_up(cp, levels=1)))

In [ ]:
# `tag` collapses many concept_ids into one readable clinical label.
# Untagged events fall back to a slug of concept_name so sequences stay legible.
print(events.tag.value_counts().head(15).to_string())
print(f"\ntagged into a codeset: {events.in_codeset.mean():.1%}")

Roughly a quarter to a third of events land in a curated codeset. That ratio is normal and
it is a useful number to report: it quantifies how much of the record your hand-built
domain knowledge actually covers. Zack's unknown-unknowns objection is exactly the other
two thirds.

*In plain terms: we only defined clinical labels (chest_pain, anxiety, panic_attack, ekg, ...) for the codes relevant to this specific question. Everything else in a patient's record still exists in the data, we just haven't given it a friendly name -- it isn't lost, and it's part of what later, more automated methods (Day 4's embeddings) can pick up on that our hand-picked labels can't.*

## 3 · The bag of codes, and the exact failure

In [ ]:
# The bag of codes, as AGE_SEX_CROSS says: 'plain' = one column per code | 'crossed' = ONLY crossed features, every code x the patient's sex x age band
# at the date of the event ('anxiety_F_35-44'; a code that happens in two bands is two columns) | 'both' = the two together | 'extended' / 'explicit' = the plain bag here (the demographic columns are added in section 4).
_vc   = events['tag'].value_counts()
codes = list(_vc[_vc >= 20].index)                         # min_count=20 events: the coarse first cut, as in bag_of_codes_sparse
if PLAIN:
    Xp, pids_p, cols_p = eo.bag_of_codes_sparse(events, min_count=20)     # sparse straight away: no dense patient x code table is ever built
    print("plain bag of codes:", Xp.shape, f"density {Xp.nnz / (Xp.shape[0] * Xp.shape[1]):.3%}")
else:
    Xp, cols_p, pids_p = None, [], np.sort(events['person_id'].unique())
if CROSS:
    Xc, cols_c = eo.age_sex_block(events, person, pids_p, codes, AGE_EDGES, min_patients=1, col='tag')     # one pass over the events
    print("crossed bag (code x sex x age band at the event):", Xc.shape, f"density {Xc.nnz / (Xc.shape[0] * Xc.shape[1]):.3%}")
else:
    Xc, cols_c = None, []

_X, _cols = (Xp, cols_p) if PLAIN else (Xc, cols_c)
display(pd.DataFrame(_X[:5, :10].toarray(), index=pids_p[:5], columns=_cols[:10]))
_i = 0      # the first patient: the columns that are on
if PLAIN:
    print(f"\npatient {pids_p[_i]}  plain  :", [cols_p[j] for j in Xp[_i].indices][:8])
if CROSS:
    print(f"patient {pids_p[_i]}  crossed:", [cols_c[j] for j in Xc[_i].indices][:8])

**Goal:** get a feel for `X` before we do anything clever with it — which codes are common, which are rare, and how that's different from what an NMF component (two sections down) will tell us.

Prevalence read directly off `X`, before any factorization -- contrasted with the NMF-loading readout two sections down. A loading tells you what's characteristic of a *component*, not how common a code is overall: `chest_pain` is ~100% prevalent here yet won't dominate any single NMF component once it's dropped as near-constant.

*In plain terms: "prevalence" here just means "what % of patients have this code at all." That's a different question from "which codes define this cluster/component" -- a code can be common across the whole cohort (like `chest_pain`, since that's how the cohort was defined) without being distinctive of any particular subgroup.*

`AGE_SEX_CROSS` decides what the bag holds. `'plain'`: one column per code. `'crossed'`: **only crossed columns**, every code crossed with the patient's **sex and age band at the date of the event** (`anxiety_F_35-44`, `blood_cell_count_aut__procedure__4035541_F_35-44`); a patient who had the code in two age bands has two columns. `'both'`: the plain and the crossed columns together. `'extended'`: the plain columns plus the patient's sex and age band at **each chest pain event** (a patient with chest pain at 38 and again at 44 has both `demo_age_35-44` and `demo_age_45-54`; sex is one column per patient), one-hot, not crossed with any code (section 4). `'explicit'`: the plain columns plus Sean's two high-risk flags, `demo_male_45plus` and `demo_female_55plus` (1 if the patient was a man aged 45+, or a woman aged 55+, at **any chest pain visit**; a man with chest pain at 44 and again at 46 is 1 in `demo_male_45plus`), one-hot, not crossed with any code (section 4). With `'crossed'` the plain code is not a column: it is kept as a reference for names and prevalence (`Xs_tok`, below).

In [ ]:
for name, X, cols in [('plain', Xp, cols_p), ('crossed (code x sex x age band at the event)', Xc, cols_c)]:
    if X is None:
        continue
    prevalence = pd.Series(np.asarray(X.mean(axis=0)).ravel(), index=cols).sort_values(ascending=False)
    print(f"{name}: {len(prevalence)} columns, prevalence range "
          f"{prevalence.min():.3%} - {prevalence.max():.3%}")
    display(prevalence.head(30).to_frame('prevalence').style.format('{:.1%}'))

    plt.figure(figsize=(7, 5))
    prevalence.head(30).iloc[::-1].plot.barh(color='#2C5F6F')
    plt.xlabel('fraction of cohort'); plt.title(f'Top 30 {name.split(" ")[0]} columns by direct prevalence')
    plt.tight_layout(); plt.show()

if CROSS:       # the crossed copies of the commonest code that is not a curated group
    _t = next(t for t in events['tag'].value_counts().index if t not in codesets and t != ANCHOR)
    print(f"crossed copies of {_t}:")
    display(prevalence[[c for c in prevalence.index if c.rsplit('_', 2)[0] == _t]].to_frame('prevalence').style.format('{:.1%}'))

In [ ]:
# Sean's code sets in the plain bag: how common each one is (% of patients with at least one event of it) and where it ranks among ALL plain columns (1 = commonest).
if Xp is None:
    print(f"AGE_SEX_CROSS = '{AGE_SEX_CROSS}': there is no plain bag, nothing to rank.")
else:
    n_all    = pd.Series(np.asarray(Xp.sum(axis=0)).ravel(), index=cols_p).sort_values(ascending=False)    # patients per plain column (whole numbers: no rounding in the ranks)
    prev_all = n_all / Xp.shape[0]                                                                          # every plain column, as in the chart above
    rows = []
    for tag in codesets:
        n = events.loc[eo.group_mask(events, tag), 'person_id'].nunique()       # patients with any event of the code set; a concept that is in two code sets counts in both
        rows.append({'code set': tag, 'concepts': len(codesets[tag]), 'prevalence': n / Xp.shape[0], 'own column': prev_all.get(tag, np.nan),
                     'rank': int((n_all > n).sum()) + 1 if n > 0 else np.nan})
    sean = pd.DataFrame(rows).set_index('code set').sort_values('prevalence', ascending=False)
    print(f"{len(prev_all):,} plain columns, rank 1 = the commonest. 'own column' is empty when the code set has no column of its own "
          f"(its concepts belong to an earlier code set, e.g. uw_noncardiac_cp -> chest_pain); 'rank' is empty when no patient has the code set.")
    display(sean.style.format({'prevalence': '{:.1%}', 'own column': '{:.1%}', 'rank': '{:,.0f}'}))

    fig, ax = plt.subplots(1, 2, figsize=(13, 5), gridspec_kw={'width_ratios': [1.1, 1]})
    s = sean.iloc[::-1]
    ax[0].barh(s.index, s['prevalence'], color='#2C5F6F')
    for y, (p, r) in enumerate(zip(s['prevalence'], s['rank'])):
        ax[0].text(p + 0.01, y, f"#{r:,.0f}" if r == r else 'none', va='center', fontsize=8)
    ax[0].set_xlim(0, 1.12); ax[0].set_xlabel('fraction of cohort')
    ax[0].set_title("Sean's code sets: prevalence (label = rank among all plain columns)", fontsize=10)
    ax[1].plot(np.arange(1, len(prev_all) + 1), prev_all.to_numpy(), color='#B8C4C9', lw=2, label=f'all {len(prev_all):,} plain columns')
    ax[1].scatter(sean['rank'], sean['prevalence'], color='#C4622D', zorder=3, label="Sean's code sets")
    for t, r in sean.dropna(subset=['rank']).iterrows():
        ax[1].annotate(t, (r['rank'], r['prevalence']), xytext=(4, 4), textcoords='offset points', fontsize=7)
    ax[1].set_xscale('log'); ax[1].set_xlabel('rank (1 = commonest column, log scale)'); ax[1].set_ylabel('fraction of cohort')
    ax[1].set_title('Where they sit on the whole prevalence curve', fontsize=10); ax[1].legend(fontsize=8)
    plt.tight_layout(); plt.show()


### Choosing `min_count` from prevalence

`bag_of_codes(min_count=...)` drops a code unless it has at least that many **events** in the
whole cohort, so one patient with 20 refills can keep a code alive. What actually limits a
model is how many **patients** carry a code, so this cell works in patients.

*In plain terms: a code seen in only a handful of patients can't form a reliable pattern, but
dropping too many codes throws away real signal. This table shows, for each cutoff, how many
columns survive and how much of the patient-by-code information is kept, then suggests a
cutoff.*

The suggested floor is the number of patients a code needs so that a training split is
expected to contain at least `MIN_TRAIN_PATIENTS` of them: `ceil(MIN_TRAIN_PATIENTS * cohort /
N_TRAIN)`. It scales with cohort size. It is a heuristic, not a statistical test, and it works
on `events`, so it does not need the full matrix (which would be too large to build with no
cutoff).

With `AGE_SEX_CROSS = 'crossed'` or `'both'` the table also counts the crossed bag of section 3 (every code x sex x age band at the event): `crossed columns` and `crossed pairs_kept_%` for the same cutoffs, and the sum with the plain columns. The floor itself does not depend on the number of columns: it is per column, so the same number of patients applies to a crossed column. Section 3 crosses only the codes with at least 20 events, so the crossed counts of the rows below 20 patients are incomplete.

The floor is applied **once**, to the columns of the bag that `AGE_SEX_CROSS` picked. A crossed column needs `MIN_PATIENTS` patients, so a code split over many age bands and sexes needs enough patients in each. The plain codes above the floor are always built, as the reference (names, prevalence, the ladder's baseline). The next cell builds them and the crossed columns above the floor, and section 4 makes the single matrix `Xs` that every later cell (distances, NMF) uses.


In [ ]:
import math

MIN_TRAIN_PATIENTS = 5            # a code should appear in >= this many patients of a training split
n_pat   = events.person_id.nunique()
N_TRAIN = min(20_000, int(0.6 * n_pat))      # training-split size used by the k-sweep cells

# patients per code, and events per code (what bag_of_codes' min_count counts)
code, tags = pd.factorize(events['tag'])
key        = events['person_id'].to_numpy(np.int64) * len(tags) + code
prev_pat   = pd.Series(np.bincount(np.unique(key) % len(tags), minlength=len(tags)), index=tags)
ev_cnt     = pd.Series(np.bincount(code, minlength=len(tags)), index=tags)
all_pairs  = int(prev_pat.sum())             # distinct (patient, code) pairs = ones in the full 0/1 matrix

def summarize(mask):
    return {'columns': int(mask.sum()),
            'pairs_kept_%': round(100 * prev_pat[mask].sum() / all_pairs, 2)}

CUTOFFS = [1, 2, 5, 10, 20, 35, 50, 100, 200, 500]
rows = [{'rule': f'>= {p} patients', 'prevalence_%': round(100 * p / n_pat, 4), **summarize(prev_pat >= p)}
        for p in CUTOFFS]
rows.append({'rule': 'current min_count=20 (events)', 'prevalence_%': np.nan, **summarize(ev_cnt >= 20)})
rule_tbl = pd.DataFrame(rows).set_index('rule')

if CROSS:       # the same rules on the crossed bag of section 3 (Xc is binary: a column sum is a number of patients)
    pc      = np.asarray(Xc.sum(axis=0)).ravel()
    pairs_c = int(pc.sum())                  # distinct (patient, code, sex, band) cells = ones in the full crossed matrix
    cross   = pd.DataFrame([{'rule': f'>= {p} patients', 'crossed columns': int((pc >= p).sum()),
                             'crossed pairs_kept_%': round(100 * pc[pc >= p].sum() / pairs_c, 2)} for p in CUTOFFS]).set_index('rule')
    rule_tbl = rule_tbl.join(cross)
    rule_tbl['plain + crossed columns'] = rule_tbl['columns'] + rule_tbl['crossed columns']

MIN_PATIENTS = math.ceil(MIN_TRAIN_PATIENTS * n_pat / N_TRAIN)
KEEP_TAGS    = prev_pat.index[prev_pat >= MIN_PATIENTS]
display(rule_tbl)
if CROSS:
    print("crossed columns come from the codes with >= 20 events (section 3), so the rows below 20 patients undercount them; from 20 patients up the rows are complete")

# the clinical (codeset) tags are the ones the analysis is about -- do any fall below the floor?
codeset_tags = [t for t in codesets if t in prev_pat.index]
lost = prev_pat[codeset_tags][lambda s: s < MIN_PATIENTS]
print(f"cohort {n_pat} patients, training split {N_TRAIN}")
print(f"suggested floor: {MIN_PATIENTS} patients ({100 * MIN_PATIENTS / n_pat:.3f}% prevalence) -> "
      f"{len(KEEP_TAGS)} of {len(prev_pat)} columns, "
      f"{100 * prev_pat[KEEP_TAGS].sum() / all_pairs:.1f}% of patient-code pairs kept")
print("codeset tags below the floor:", lost.to_dict() if len(lost) else "none")
if CROSS:       # what the same floor leaves of the crossed bag
    _tk = np.array([nm.rsplit('_', 2)[0] for nm in cols_c])
    _kc = (pc >= MIN_PATIENTS) & (_tk != ANCHOR)
    print(f"crossed columns kept at the same floor ({MIN_PATIENTS} patients): {int(_kc.sum()):,} of {len(cols_c):,}")
    print(f"crossed pairs kept: {100 * pc[_kc].sum() / pairs_c:.1f}% of the (patient, code, sex, band) pairs")
n_fit = (len(KEEP_TAGS.difference([ANCHOR])) if PLAIN else 0) + (int(_kc.sum()) if CROSS else 0)
print(f"AGE_SEX_CROSS = '{AGE_SEX_CROSS}': matrix to fit {n_fit:,} columns{' + the demographic columns of section 4' if DEMO else ''}")

fig, ax = plt.subplots(figsize=(7, 3.2))
ps = np.unique(np.round(np.geomspace(1, max(2, prev_pat.max() / 5), 40)).astype(int))
ax.plot(ps, [(prev_pat >= p).sum() for p in ps], color='C0')
ax.set_xscale('log'); ax.set_xlabel('minimum patients per code'); ax.set_ylabel('columns kept', color='C0')
ax2 = ax.twinx()
ax2.plot(ps, [100 * prev_pat[prev_pat >= p].sum() / all_pairs for p in ps], color='C1')
ax2.set_ylabel('% of patient-code pairs kept', color='C1')
ax.axvline(MIN_PATIENTS, ls='--', c='grey')
ax.set_title('what each prevalence floor costs')
plt.tight_layout(); plt.show()

print("\nTo apply it, build the matrix from the kept codes only (min_count=1 on pre-filtered events):")
print("    Xsp, person_ids, all_cols = eo.bag_of_codes_sparse(events[events.tag.isin(KEEP_TAGS)], min_count=1)")

In [ ]:
Xsp, person_ids, all_cols = eo.bag_of_codes_sparse(events[events.tag.isin(KEEP_TAGS)], min_count=1)   # the plain codes above the floor (fitted for 'plain' and 'both', a reference for 'crossed')
print("plain codes above the floor:", Xsp.shape)

if CROSS:       # the same floor on the crossed bag of section 3: a crossed column needs >= MIN_PATIENTS patients, and its code must be above the floor too
    assert np.array_equal(pids_p, person_ids), 'the crossed bag and the plain matrix must have the same patients in the same order'
    _n    = np.asarray(Xc.sum(axis=0)).ravel()
    _tok  = set(all_cols)
    _keep = [j for j, nm in enumerate(cols_c) if _n[j] >= MIN_PATIENTS and nm.rsplit('_', 2)[0] in _tok]
    Xc_kept, cols_c_kept = Xc[:, _keep].tocsr(), [cols_c[j] for j in _keep]
    print(f"crossed features above the floor: {Xc_kept.shape}  ({len(_keep):,} of the {len(cols_c):,} crossed columns, >= {MIN_PATIENTS} patients each)")
else:
    Xc_kept, cols_c_kept = None, []

In [ ]:
# Two dx_9487 patients with a near-IDENTICAL bag of codes and opposite triage answers.
weak = eo.chest_pain_labels(events)                 # rule-based probe labels: psych vs cardiac
lab  = weak.set_index('person_id').chest_pain_label

_eg = events[events.groups.notna()]
Xg = eo.bag_of_codes(_eg.assign(group=_eg.groups.astype(str).str.split('|')).explode('group').reset_index(drop=True), col='group', min_count=1)   # group-level checklist (a concept counts in EVERY group it is in), demo only
both = [p for p in person_ids
        if Xg.loc[p].get('chest_pain', 0) == 1
        and Xg.loc[p].get('ekg', 0) == 1
        and p in lab.index]

a = next((p for p in both if lab[p] == 'psych'),   None)
b = next((p for p in both if lab[p] == 'cardiac'), None)
assert a and b, ("No matching-bag pair found in this sample. Widen the `both` filter, "
                 "or inspect `both` to see which side is missing.")

print("bag-of-codes view -- indistinguishable on chest_pain + ekg alone:")
display(Xg.loc[[a, b], ['chest_pain', 'ekg']])

print("\nactual timelines -- obviously different:")
FOCUS = ['chest_pain', 'ekg', 'anxiety', 'gerd', 'troponin', 'uw_cardiac_testing', 'uw_cad_dx',
         'uw_abnormal_ekg', 'nitrates']
for pid in (a, b):
    s = events[(events.person_id == pid) & eo.group_mask(events, FOCUS)]
    print(f"\n  person_id {pid}  ({lab[pid]})")
    print(s[['event_date', 'groups', 'tag']].drop_duplicates('tag').to_string(index=False))

**That is the motivation for the whole week.** Near-identical features, opposite triage
answer. The information lives in the order and the surrounding evidence, and a flat bag of
codes throws both away.

This is not a modelling failure. No estimator can recover something the input does not
contain.

*In plain terms: this isn't a case of "use a fancier algorithm and it'll figure it out." The bag-of-codes table literally does not contain the information needed to tell these two patients apart -- both rows are 1, 1. No amount of machine learning can pull a distinction out of data where the distinction was never recorded in the first place. You have to fix the representation, not the model.*

### On `chest_pain_labels()`

There is no adjudicated ground truth for "was this actually cardiac." `chest_pain_labels()`
builds probe labels from **corroborating co-evidence only** — anxiety/panic diagnoses,
SSRIs, or benzodiazepines on the `psych` side; troponin, stress tests, coronary
angiography, or a CAD diagnosis on the `cardiac` side. Patients with both kinds of
evidence, or neither, are dropped, and the counts are printed so you can see how many.

*In plain terms: instead of guessing the reason for the chest pain directly, we look for other, more specific evidence nearby — did the patient also have an anxiety/panic diagnosis or an SSRI/benzodiazepine prescription? Did they also have a troponin, stress test, or coronary angiography? If the evidence points clearly one way, we use that as the label. If it points both ways, or neither way, we honestly say "we don't know" and drop them rather than force a guess.*

**The time order is deliberately not used here.** If you labelled by "anxiety dx before
chest pain code" and then tested whether the order predicts the label, you'd get perfect
accuracy by construction and learn nothing. Days 3-4 test order directly — the label has to
stay independent of it, or the test is circular.

*In plain terms: it would be tempting to say "anxiety diagnosis came before the chest pain code, so it must be psych-related." But that's using the very thing we're trying to test (timing) to build the answer key -- then "discovering" that timing predicts the answer would be circular, not a real finding. So the answer key is built from a different, independent kind of evidence, and timing is only used afterward to see if it can predict that answer.*

In [ ]:
from sklearn.metrics import pairwise_distances
import scipy.sparse as sp

# chest_pain is the cohort anchor -- ~100% prevalence here, so it's a near-
# constant column that carries no discriminative signal and would just eat a
# component. Drop it before any distance/NMF fit; X itself (with the
# column) stays untouched for the two-patient demo above and Table 1 below.
# Everything downstream works on a SPARSE copy (Xs): same values as the dense matrix without the
# anchor column, a fraction of the memory, and faster NMF fits. No dense copy without
# `chest_pain` is ever built.
# ONE matrix from here on, as AGE_SEX_CROSS says: 'plain' = the plain codes above the floor | 'crossed' = only the crossed features above the floor | 'both' = the two | 'extended' = plain + the sex and age band one-hots | 'explicit' = plain + the two risk flags (man aged 45+, woman aged 55+). The anchor is out (its crossed copies too).
# The plain codes (TOK_COLS, Xs_tok) are always kept as the reference: names of bundles, prevalence, and the ladder's baseline.
_all     = pd.Index(all_cols)
_anchor  = set(events.loc[eo.group_mask(events, 'chest_pain'), 'tag'])
TOK_COLS = [c for c in _all if c not in _anchor]                  # the plain codes: what the AHA k=8 notebook fits on; the bundle tables name codes, not crossed columns
n_pat    = Xsp.shape[0]
Xs_tok   = Xsp[:, _all.get_indexer(TOK_COLS)].astype(float).tocsr()
cp_events = events.loc[eo.group_mask(events, ANCHOR)]                                            # every chest pain event
first_cp  = cp_events.groupby('person_id')['event_date'].min().rename('ref_date').to_frame()
demo      = eo.demographics(person, first_cp, AGE_EDGES)      # read-out only: sex and age band at the FIRST chest pain, the stratum of Part A+ and the ladder
BLOCKS   = {}                                                     # block -> (matrix, column names), rows aligned with person_ids
if CROSS:
    _ok = [j for j, nm in enumerate(cols_c_kept) if nm.rsplit('_', 2)[0] not in _anchor]        # the anchor's own crossed copies go too
    BLOCKS['S'] = (Xc_kept[:, _ok].tocsr(), [cols_c_kept[j] for j in _ok])
BLOCKS['D'] = eo.demographic_block(cp_events, person, person_ids, AGE_EDGES, min_patients=MIN_PATIENTS)    # sex + the age band AT EACH chest pain event, one-hot; always built (a few columns), fitted only in 'extended', and a row of the ladder
BLOCKS['R'] = eo.risk_age_sex_block(cp_events, person, person_ids, RISK_RULES, min_patients=MIN_PATIENTS)    # Sean's two flags, 1 if ANY chest pain visit meets the rule; always built (two columns), fitted only in 'explicit', and a row of the ladder
WEIGHT   = {'S': W_S, 'D': W_D, 'R': W_R}
assert not ({n for _, names in BLOCKS.values() for n in names} & set(TOK_COLS)), 'a crossed column has the name of a plain column'

def assemble(blocks, plain=True):
    """Plain codes (when plain=True) + the named blocks -> (CSR matrix, column names)."""
    mats  = ([Xs_tok] if plain else []) + [WEIGHT[b] * BLOCKS[b][0] for b in blocks]
    names = (TOK_COLS if plain else []) + [n for b in blocks for n in BLOCKS[b][1]]
    return sp.hstack(mats, format='csr'), names

FIT = [b for b, on in (('S', CROSS), ('D', AGE_SEX_CROSS == 'extended'), ('R', AGE_SEX_CROSS == 'explicit')) if on]       # the blocks AGE_SEX_CROSS puts in the fitted matrix; the others stay for the ladder
Xs, nmf_cols = assemble(FIT, plain=PLAIN)      # the matrix the distances and the NMF use
print(f"Xs {Xs.shape}: '{AGE_SEX_CROSS}' bag, density {Xs.nnz / (Xs.shape[0] * Xs.shape[1]):.3%}   (plain codes above the floor, the reference: {Xs_tok.shape[1]:,})")

def code_H(components):
    """NMF components (k x len(nmf_cols)) -> k x codes: the loadings of a code's plain column and of its crossed columns (sex, band) are summed, so tables can name codes. The demographic columns of 'extended' and 'explicit' are not codes: they drop out here (Part A+ reads them)."""
    H = pd.DataFrame(components, columns=nmf_cols)
    tok = set(TOK_COLS)
    H = H.T.groupby(pd.Series([n if n in tok else n.rsplit('_', 2)[0] for n in nmf_cols], index=nmf_cols)).sum().T
    return H[[c for c in TOK_COLS if c in H.columns]]

def dist_to_all(row, M, metric, chunk=1_000):   # 10,000 dense rows would not fit once the crossed columns are in
    """Distances from one row to every row of sparse M, densifying only `chunk` rows at a time."""
    return np.concatenate([pairwise_distances(row, M[i:i + chunk].toarray(), metric=metric)[0]
                           for i in range(0, M.shape[0], chunk)])

sub = Xs[:400].toarray()
for m in [#'euclidean', 
          'cosine', 'jaccard']:
    D  = pairwise_distances(sub, metric=m)
    iu = np.triu_indices(len(sub), 1)
    print(f"{m:10s} mean={D[iu].mean():.3f}  sd={D[iu].std():.3f}")

pos = list(person_ids).index(a)
print()
for m in [#'euclidean', 
          'cosine', 'jaccard']:
    D  = dist_to_all(Xs[pos:pos+1].toarray(), Xs, m)
    nn = [person_ids[i] for i in np.argsort(D)[1:6]]
    print(f"{m:10s} neighbours of {a}: {[lab.get(p, '?') for p in nn]}")

**Rule of thumb for claims: use cosine on binary or count code vectors.** Euclidean punishes
patients simply for having long histories, so the distance ends up driven by utilisation
volume rather than clinical content — which in your data is partly a function of what got
ordered during underwriting.

*In plain terms: a patient with 200 recorded codes isn't automatically "less similar" to everyone else than a patient with 20 codes — but a naive distance metric (Euclidean) treats it that way, because it's sensitive to how many codes are turned on, not just which ones. Cosine distance cares about the pattern of codes, not the raw count, which is what we actually want to compare.*

## 5 · The curse of dimensionality

On the full Ethos vocabulary you will have tens of thousands of columns, not the few dozen
here. At that width every pair of patients becomes equidistant and "nearest neighbour" stops
meaning anything.

*In plain terms: with only a handful of columns, "similar" and "different" are easy to tell apart. Once you have tens of thousands of columns and each patient only has a handful of them turned on, almost every pair of patients ends up looking roughly equally (dis)similar to each other -- there's no longer a meaningful "close" versus "far." This is a general math fact (the "curse of dimensionality"), not something specific to our data.*

In [ ]:
rng = np.random.default_rng(0)
print(f"{'dims':>6} {'min/max distance ratio':>24}")
for d in [10, 50, 200, 1000, 5000]:
    Zr = (rng.random((200, d)) < 0.02).astype(float)     # sparse, like code data
    D  = pairwise_distances(Zr)
    iu = np.triu_indices(200, 1)
    print(f"{d:>6} {D[iu].min() / D[iu].max():>24.3f}")
print("\nRatio -> 1 means every patient is equally far from every other.")

## 5b · Age and sex at exposure: what is in the matrix

`AGE_SEX_CROSS` picked what the bag of codes holds, and `Xs` is what is left after the prevalence floor (applied once, above): `'plain'` the plain codes, `'crossed'` only the crossed features, `'both'` the two together, `'extended'` the plain codes plus the sex and age band one-hots (D), `'explicit'` the plain codes plus the two risk flags (R). A crossed feature is a code (a curated group or a concept-level code) copied into the sex and age band of the patient **at the date of each event**: `anxiety_F_35-44`, `blood_cell_count_aut__procedure__4035541_F_35-44`. Age is `(event date - birth date) / 365.25`, cut into 10-year bands (`<25`, `25-34`, `35-44`, `45-54`, `55-64`, `65+`; `AGE_EDGES` sets them); sex is the person's own. A patient with anxiety at 38 and again at 48 has both `anxiety_F_35-44` and `anxiety_F_45-54`.

Sex does not change, so with `'crossed'` a woman and a man never share a column: expect components that are female or male versions of a care pattern, and read the NMI in Part A+ and the ladder (`'both'` keeps the plain columns, which both sexes share). The plain codes are always a reference (`TOK_COLS`, `Xs_tok`): bundle tables, the components json and the group loadings fold the loadings back to codes (`code_H`: plain and crossed columns of a code summed). A crossed name splits back with `name.rsplit('_', 2)` into code, sex and band.

The first version of this notebook also had a sex and age one-hot block (D) and recency windows (R). In the full run both dominated the components (top columns `hba1c@48m`, `troponin@48m`; two components that were just `sex=F` and `sex=M`), so they came out; D is back on its own as `'extended'`, with a weight `W_D`, to test it cleanly; recency waits for the application date.

This section only reads the matrix. `demo` (sex and age band at the **first chest pain**) is for reading the results, who is in each component. It is never a feature. The D features of `'extended'` come from the chest pain events themselves: the age band at **each** chest pain event, so a patient with chest pain at 38 and again at 44 has `demo_age_35-44` and `demo_age_45-54`, plus one sex column (`demo_sex_F`, weight `W_D`). The R features of `'explicit'` are `demo_male_45plus` and `demo_female_55plus` (weight `W_R`): 1 if **any** chest pain visit is on or after the 45th birthday of a man or the 55th birthday of a woman (`RISK_RULES`).

*In plain terms: with `'crossed'` every kind of care is split by the sex and 10-year age band the patient was in when it happened, and the bundles are found on those split features, so a bundle can say "blood count at 35-44 in women".*

In [ ]:
# `demo` (sex and age band at the first chest pain) is built in section 4: a read-out only
LABEL = {'S': 'S code x sex x age band at the event', 'D': 'D sex + age band at each chest pain event (one-hot)', 'R': 'R risk flags: man aged 45+, woman aged 55+ at any chest pain visit (one-hot)'}
rows = [{'block': 'plain codes' + (' (reference, not fitted)' if not PLAIN else ''), 'columns': Xs_tok.shape[1], 'non-zeros': Xs_tok.nnz, 'weight': 1.0}]
rows += [{'block': LABEL[b] + ('' if b in FIT else ' (built for the ladder, not fitted)'), 'columns': M.shape[1], 'non-zeros': M.nnz, 'weight': WEIGHT[b]} for b, (M, _) in BLOCKS.items()]
display(pd.DataFrame(rows).set_index('block'))
print(f"matrix the NMF is fitted on: {Xs.shape}, density {Xs.nnz / (Xs.shape[0] * Xs.shape[1]):.3%}, about {(Xs.data.nbytes + Xs.indices.nbytes + Xs.indptr.nbytes) / 1e9:.1f} GB")

if 'S' in BLOCKS:       # how the S cells spread over sex and age band (patient x code x band): the size of the tails shows here
    _s = pd.DataFrame([n.rsplit('_', 2) for n in BLOCKS['S'][1]], columns=['code', 'sex', 'band'])
    _s['cells'] = np.asarray(BLOCKS['S'][0].sum(axis=0)).ravel()
    print("\nS cells (patient x code x band) by sex and age band at the event:")
    display(_s.pivot_table(index='band', columns='sex', values='cells', aggfunc='sum', fill_value=0).reindex(eo.age_band_labels(AGE_EDGES)).dropna(how='all'))

## 6 · NMF

**Goal:** compress thousands of sparse code columns into a handful of components per patient. NMF only allows non-negative parts, so each component reads as a set of co-occurring codes you can name.

*In plain terms: the wide, sparse `X` table (thousands of mostly-zero columns) is summarised into a few dozen themes, each a bundle of care that tends to happen together in the same patients.*

### NMF and Choosing k for the NMF

The k=12 above was chosen by reading the components. This sweep adds numbers for it, the
NMF counterpart of Day 2's elbow + bootstrap-ARI table for KMeans.

*In plain terms: fit the NMF at several values of k and, for each one, check how well it
explains patients it was never fit on, and whether the same bundles come back on re-draws. Then pick a k where adding
more components stops paying for itself but the bundles are still reproducible.*

Every k is fit on the same `N_SWEEP` training patients and scored on a disjoint `N_HELD`
held-out set, so the comparison is fair and the error is not just training fit.

The whole sweep is repeated on every split in `SPLIT_SEEDS` (independent random train / held-out
splits of the cohort), each with `N_BOOT_K` bootstrap refits per k. More refits only sharpen the
estimate within one split. Several splits show whether a k looks stable only because of the
patients it happened to see. Every NMF fit in this section uses the same settings (`NMF_KW`,
defined in the stability cell), so the base fit and its refits are converged the same way.

The fits run in parallel (`N_JOBS` workers, because sklearn's NMF uses one core per fit) on a
sparse copy of the matrix, `Xs`. The result is the same as the dense serial version; it is only
faster and much smaller in memory.

In [ ]:
from sklearn.decomposition import NMF
from scipy.optimize import linear_sum_assignment
from sklearn.metrics import adjusted_rand_score
import time

# one place for the NMF fit settings, used by every NMF fit in this section
NMF_KW = dict(init='nndsvd', random_state=0, max_iter=2000, tol=1e-4)

N_BOOT        = 3 if SMOKE else 10   # refits; each is a full NMF fit, raise once you know the runtime
N_BOOT_LOAD   = 3 if SMOKE else 10   # refits of the stability cell below (~2 s each)
N_BOOT_SAMPLE = 20_000    # patients per resample (capped at the cohort size)
N_EVAL        = 20_000    # fixed patients used to compare argmax labels across refits

def _unit(M):
    return M / np.linalg.norm(M, axis=1, keepdims=True)

def nmf_stability(nmf_base, Xv, n_boot=N_BOOT, n_sample=N_BOOT_SAMPLE,
                  n_eval=N_EVAL, seed=0, pct_cols=None):
    """Bootstrap-refit NMF; return per-component matched cosine and per-boot ARI.

    Component order is arbitrary between fits (label switching), so each refit's
    components are matched to the base fit's by cosine similarity (Hungarian
    algorithm). ARI compares argmax patient partitions, which needs no matching.

    pct_cols (column indices) also returns, for every refit, the share (%) of each matched component's
    total loading that sits on those columns: an array (n_boot, k, len(pct_cols)), component j in the
    base fit's order. None when pct_cols is None.
    """
    k    = nmf_base.n_components_
    Hb   = _unit(nmf_base.components_)
    rng  = np.random.default_rng(seed)
    ev   = rng.choice(Xv.shape[0], min(n_eval, Xv.shape[0]), replace=False)
    lab0 = nmf_base.transform(Xv[ev]).argmax(1)
    cos, ari, pct = [], [], []
    for b in range(n_boot):
        t0  = time.perf_counter()
        idx = rng.choice(Xv.shape[0], min(n_sample, Xv.shape[0]), replace=True)
        m   = NMF(n_components=k, **NMF_KW).fit(Xv[idx])
        S        = Hb @ _unit(m.components_).T         # base x refit cosine
        rows, cols = linear_sum_assignment(-S)          # rows come back sorted 0..k-1
        cos.append(S[rows, cols])                       # cos[b][j] = base component j
        ari.append(adjusted_rand_score(lab0, m.transform(Xv[ev]).argmax(1)))
        if pct_cols is not None:
            Hm = m.components_[cols]                         # refit components in the base order
            pct.append((100 * Hm[:, pct_cols] / np.maximum(Hm.sum(axis=1, keepdims=True), 1e-12)).astype(np.float32))
        if (b + 1) % max(1, n_boot // 10) == 0:
            print(f"  boot {b + 1}/{n_boot}  {time.perf_counter() - t0:.0f}s  "
                  f"ARI {ari[-1]:.3f}")
    return np.array(cos), np.array(ari), (np.array(pct) if pct_cols is not None else None)

In [ ]:
import time
import io, contextlib
import os, math
import scipy.sparse as sp
from joblib import Parallel, delayed
from scipy import stats

K_GRID      = [4, 6, 8] if SMOKE else list(range(3, 15))
SPLIT_SEEDS = (1, 2) if SMOKE else (1, 2, 3, 4, 5)   # independent train / held-out splits of the cohort
N_SWEEP     = min(20_000, int(0.6 * Xs.shape[0]))      # train patients every k is fit on
N_HELD      = min(10_000, Xs.shape[0] - N_SWEEP)      # disjoint held-out patients for the error
N_BOOT_K    = 3 if SMOKE else 10          # bootstrap refits per k and split (reuses nmf_stability above)
MIN_SPLITS_OK = math.ceil(0.8 * len(SPLIT_SEEDS))   # a k must pass on this many splits (4 of 5)
N_JOBS      = max(1, (os.cpu_count() or 2) - 2)   # sklearn's NMF is single-threaded: one fit per core

def rel_err(X, W, H):
    """||X - W H||_F / ||X||_F without building the dense W @ H (X may be sparse)."""
    x2    = X.multiply(X).sum() if sp.issparse(X) else (X ** 2).sum()
    cross = np.sum((X @ H.T) * W)
    quad  = np.sum((W.T @ W) * (H @ H.T))
    return float(np.sqrt(max(x2 - 2 * cross + quad, 0.0) / x2))

def sweep_job(split_seed, k, Xtr, Xho, n_boot):
    """All metrics for one (split, k): one base fit plus n_boot bootstrap refits."""
    t0 = time.perf_counter()
    m  = NMF(n_components=k, **NMF_KW).fit(Xtr)
    Wh = m.transform(Xho)
    lab_ho = Wh.argmax(1)
    with contextlib.redirect_stdout(io.StringIO()):          # silence per-boot prints
        cos, ari, _ = nmf_stability(m, Xtr, n_boot=n_boot, n_sample=Xtr.shape[0],
                                 n_eval=Xtr.shape[0], seed=k)
    share = np.bincount(lab_ho, minlength=k) / len(lab_ho)
    return {'split': split_seed, 'k': k,
            'held_out_err': rel_err(Xho, Wh, m.components_),
            'ari_mean': ari.mean(), 'ari_sd': ari.std(ddof=1),
            'ari_se': ari.std(ddof=1) / np.sqrt(n_boot),
            'cos_mean': cos.mean(), 'n_unstable': int((cos.mean(0) < .75).sum()),
            'n_used': int((share >= .01).sum()), 'secs': time.perf_counter() - t0}

def make_split(split_seed):                                   # sliced once here, so workers get small inputs
    perm   = np.random.default_rng(split_seed).permutation(Xs.shape[0])
    tr, ho = perm[:N_SWEEP], perm[N_SWEEP:N_SWEEP + N_HELD]
    return Xs[tr], Xs[ho]

splits  = {s: make_split(s) for s in SPLIT_SEEDS}
jobs    = [(s, k) for k in sorted(K_GRID, reverse=True) for s in SPLIT_SEEDS]   # slow (large k) first
results = Parallel(n_jobs=N_JOBS, verbose=5)(
              delayed(sweep_job)(s, k, *splits[s], N_BOOT_K) for s, k in jobs)
res       = pd.DataFrame(results)
per_split = {s: res[res.split == s].drop(columns=['split', 'secs']).set_index('k').sort_index()
             for s in SPLIT_SEEDS}
print(f"{len(jobs)} jobs on {N_JOBS} workers; summed fit time {res.secs.sum() / 60:.1f} CPU-min")

# one row per k, summarised over splits
g     = pd.concat(per_split, names=['split']).groupby('k')
sweep = pd.DataFrame({'held_out_err': g.held_out_err.mean(),
                      'ari_mean': g.ari_mean.mean(), 'ari_sd_splits': g.ari_mean.std(),
                      'n_splits_solid': g.ari_mean.apply(lambda a: int((a >= .75).sum())),
                      'n_splits_sugg':  g.ari_mean.apply(lambda a: int((a >= .60).sum())),
                      'cos_mean': g.cos_mean.mean(), 'n_unstable': g.n_unstable.median(),
                      'n_used': g.n_used.min()})
display(sweep.round(3))

t_crit = stats.t.ppf(0.975, N_BOOT_K - 1)    # few refits -> t interval, not 1.96
fig, ax = plt.subplots(1, 2, figsize=(9.5, 3.2))
ax[0].plot(sweep.index, sweep.held_out_err, marker='o')
ax[0].set_title('held-out relative error (look for the elbow)')
for i, s in enumerate(SPLIT_SEEDS):                          # small x-offset so splits do not overlap
    d = per_split[s]
    ax[1].errorbar(d.index + (i - (len(SPLIT_SEEDS) - 1) / 2) * 0.1, d.ari_mean,
                   yerr=t_crit * d.ari_se, fmt='o', ms=3, capsize=2, alpha=.6, label=f'split {s}')
ax[1].plot(sweep.index, sweep.ari_mean, 'k-', lw=2, label='mean of splits')
ax[1].axhline(0.75, ls='--', c='green');  ax[1].axhline(0.60, ls='--', c='orange')
ax[1].set_title('bootstrap ARI per split (95% CI)'); ax[1].legend(fontsize=7, ncol=2)
for a in ax: a.set_xlabel('k'); a.set_xticks(sweep.index)
plt.tight_layout(); plt.show()

ok     = sweep[(sweep.n_splits_sugg >= MIN_SPLITS_OK) & (sweep.cos_mean >= .75)]
strong = sweep[(sweep.n_splits_solid >= MIN_SPLITS_OK) & (sweep.n_unstable == 0)]
print(f"largest k with ARI >= 0.60 on {MIN_SPLITS_OK}+ of {len(SPLIT_SEEDS)} splits and mean cos >= 0.75:",
      int(ok.index.max()) if len(ok) else "none")
print(f"largest k with ARI >= 0.75 on {MIN_SPLITS_OK}+ of {len(SPLIT_SEEDS)} splits and no unstable component (median):",
      int(strong.index.max()) if len(strong) else "none")

### Reading the k sweep

- **Held-out error** always falls as k grows, so the *elbow* is what matters, not the minimum.
- **ARI / cos** are the same bootstrap checks as above, now per k and per split. `ari_mean` is
  the average over splits and `ari_sd_splits` its spread; `n_splits_solid` / `n_splits_sugg` count
  the splits with ARI at or above 0.75 / 0.60. `n_unstable` is the median over splits. Stability
  usually decays as k rises; the largest k that still passes is an upper bound.
- **Same ranking on most splits:** the pattern belongs to k, not to the patients drawn. Prefer
  the largest k that is solid on at least 4 of 5 splits (`MIN_SPLITS_OK`). **Splits disagree
  (large `ari_sd_splits`), or neighbouring k overlap within their 95% intervals:** stability
  cannot separate them, so choose on clinical readability and keep the "suggestive" caveat on
  any k below 0.75.
- Compare k by the interval (`ari_se`), not by `ari_sd`. `ari_sd` is the spread across refits
  and does not shrink with more refits; `ari_se` does.
- **`n_used`** counts components that are the dominant component for at least 1% of held-out
  patients. A k where `n_used` stops growing is adding redundant or dead components.
- No single column picks k. A defensible k sits **after the error elbow and before stability
  collapses**. The final check is still
  whether the extra components read as distinct clinical bundles, as in the k=12 discussion.

### Are those 11 components stable? Bootstrap validation of the NMF
Now let's focus on looking stability within K=11
*In plain terms: re-run the NMF on random re-draws of the patients and see whether the same
code bundles show up. A bundle that keeps reappearing is real structure; one that only
appears for some draws is an accident of which patients were sampled.*

NMF component numbering is arbitrary between fits (component 3 in one run may be component 7
in the next), so each refit's components are first **matched** to the base fit's by cosine
similarity. Two outputs:

1. **Per component:** matched cosine between the base bundle and its refit counterpart.
2. **Cohort level:** ARI between the `argmax` patient labels, comparable with Day 2.

### Reading the stability table for K=11

- **`cos_mean` near 1** means the component's code bundle reappears almost unchanged in every
  resample. **Low `cos_mean` or high `cos_sd`** means the bundle depends on which patients
  happened to be drawn, so don't build a rule on it.
- The cosine cut-offs (0.90 solid, 0.75 suggestive) are working heuristics, not standards.
- **Bootstrap ARI** compares the `argmax` patient labels from each refit with the base
  labels, so it is directly comparable with Day 2's KMeans bootstrap ARI (0.75 solid, 0.60
  suggestive). A component can be stable as a *bundle* while the `argmax` *labels* are
  noisy, because soft loadings tie between bundles. High cos with low ARI is that case.
- Each refit sees only `N_BOOT_SAMPLE` patients, fewer than the base fit's full cohort, so
  this is a slightly pessimistic stability estimate.
- This validates stability, not meaning. Whether component 2 really is "cardiac workup" is
  still the read-the-top-codes judgement above.

### Final Bundles for K=11

In [ ]:
# NMF factorises into non-negative PARTS, so each component reads as a set of
# co-occurring codes. Unlike SVD components you can name them.
#
# In plain terms: NMF is only allowed to add codes together, never subtract
# them, so each component ends up looking like "a bundle of codes that tend
# to show up together" -- e.g. "anxiety + benzodiazepine" reads as an actual
# clinical bundle, not an abstract direction in space. That's why we print
# the top-10 codes per component below: it's how you check whether a
# component actually means something clinically, or is noise.

for k in [K_OVERRIDE or int(strong.index.max())]:
#for k in [11]:
    nmf = NMF(n_components=k, **NMF_KW).fit(Xs)
    H   = pd.DataFrame(nmf.components_, columns=nmf_cols)
    lab_k = nmf.transform(Xs).argmax(axis=1)
    dom = pd.Series(lab_k).value_counts().sort_index()
    # save the fit for reuse (Day 3 loads it instead of refitting); the tag keeps cohorts apart
    np.savez(f"nmf_k{k}_{TAG}.npz", H=H.to_numpy(), cols=np.array(H.columns, dtype=str),
             pids=np.asarray(person_ids), labels=lab_k)
    print (f"\nNMF with {k} components:")
    for c in range(k):
        n    = int(dom.get(c, 0))
        row  = H.iloc[c]
        top  = row.sort_values(ascending=False).head(10)
        share = (top / row.sum() * 100).round(1)
        # codes = [f"{name} ({pct}%)" for name, pct in share.items()]
        codes = [f"{name}" for name, pct in share.items()]
        print(f"component {c} (n={n}, {n / n_pat * 100:.1f}%): {codes}")

**NMF bundles as a table** — one row per component for the underwriter review: n (%), the top-5 OMOP entities (name, vocabulary + code, concept_id, share of the bundle's patients) and two empty columns to fill in (pathway, UW suggestion). Written to `nmf_components_<cohort>_k<k>_v3.xlsx`. `RANK_BY = 'prevalence'` ranks by how many of the bundle's patients have the code instead of by NMF loading.

In [ ]:
COHORT = 'chestpain'
# Export the NMF components as a table to review with the underwriters: one row per bundle, the last two columns are left empty to fill in.
import re
try:
    import openpyxl
except ImportError as e:
    raise ImportError("pip install openpyxl in this kernel's environment, then rerun this cell") from e
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

RANK_BY = 'loading'        # 'loading' = codes that define the bundle (NMF loading); 'prevalence' = codes most patients of the bundle have
TOP_N   = 5
k_exp   = nmf.n_components_
import os; os.makedirs('outputs', exist_ok=True)
XLSX_NMF = f"outputs/nmf_components_{COHORT}_k{k_exp}_{VARIANT}.xlsx"

lab_e  = nmf.transform(Xs).argmax(1)                                   # dominant component per patient (same rule as the sizes above)
Hc     = code_H(nmf.components_)                                      # codes, not crossed columns: the crossed loadings of a code are summed over sex and band
meta   = events.drop_duplicates('tag').set_index('tag')

# TOKEN_LEVEL == 'group': a token can be a codeset tag that collapses several concepts, so there is no single concept_id to print.
# Show the tag, how many concepts it holds and its most frequent member. Concept-level and untagged tokens keep the usual text.
# (_n_members, _is_group_token, _top_member are reused by the components-json cell below.)
_n_members = events[['tag', 'concept_id']].drop_duplicates().groupby('tag').size()
def _is_group_token(tok):
    return TOKEN_LEVEL == 'group' and tok in codesets and _n_members.get(tok, 0) > 0
def _top_member(tok):
    sub = events.loc[events['tag'] == tok, ['concept_id', 'concept_name', 'vocabulary_id', 'concept_code']]
    return sub[sub['concept_id'] == sub['concept_id'].value_counts().index[0]].iloc[0]
def _group_entity(tok, prev):
    m, n = _top_member(tok), int(_n_members[tok])
    return f"{tok} [codeset: {n} concept{'s' if n != 1 else ''}; most frequent {m.vocabulary_id} {m.concept_code} {str(m.concept_name)[:40]}] ({prev:.0%} of bundle)"

def entity(tok, prev):
    if _is_group_token(tok):
        return _group_entity(tok, prev)
    r = meta.loc[tok]
    name = re.split(r' \(|;| This ', str(r.concept_name))[0].strip()
    name = name if len(name) >= 8 else str(r.concept_name)          # packs like '{6 (azithromycin ...) } Pack [Z-PAK]' would be cut to '{6'
    name = name if len(name) <= 60 else name[:60].rsplit(' ', 1)[0] + '…'
    return f"{name} [{r.vocabulary_id} {r.concept_code}] (concept_id {int(r.concept_id)}; {prev:.0%} of bundle)"

rows_e = []
for c in range(k_exp):
    members = np.flatnonzero(lab_e == c)
    prev  = pd.Series(np.asarray(Xs_tok[members].mean(axis=0)).ravel(), index=TOK_COLS)           # share of the bundle's patients with each code (plain codes)
    score = Hc.iloc[c] if RANK_BY == 'loading' else prev
    top   = score.nlargest(TOP_N).index
    rows_e.append({'Group (bundle)': f"c{c}",
                   'n (%)': f"{len(members):,} ({len(members) / len(lab_e):.1%})",
                   'Most frequently OMOP clinical entities found': "\n".join(f"{i}. {entity(t, prev[t])}" for i, t in enumerate(top, 1)),
                   'Suggested Clinical Pathway': '',
                   'UW suggestion': ''})
nmf_table = pd.DataFrame(rows_e)

F = lambda **kw: Font(name='Arial', size=10, **kw)
thin = Side(style='thin', color='D0D5DB')
wb = openpyxl.Workbook(); ws = wb.active; ws.title = f'NMF k={k_exp}'
ws.append(list(nmf_table.columns))
for j in range(1, ws.max_column + 1):
    h = ws.cell(1, j); h.font = Font(name='Arial', size=10, bold=True, color='FFFFFF')
    h.fill = PatternFill('solid', fgColor='334E68'); h.alignment = Alignment(wrap_text=True, vertical='center', horizontal='center')
ws.row_dimensions[1].height = 30
for row in nmf_table.itertuples(index=False):
    ws.append(list(row))
for i in range(2, len(nmf_table) + 2):
    ws.row_dimensions[i].height = 15 * TOP_N + 6
    for j in range(1, ws.max_column + 1):
        c_ = ws.cell(i, j); c_.font = F(bold=(j == 1)); c_.border = Border(bottom=thin)
        c_.alignment = Alignment(wrap_text=True, vertical='top', horizontal='center' if j in (1, 2) else 'left')
for j, w in enumerate([14, 16, 95, 40, 40], 1):
    ws.column_dimensions[get_column_letter(j)].width = w
ws.freeze_panes = 'B2'
ws.cell(len(nmf_table) + 3, 1, f"Source: Day 1 v3 {COHORT}, NMF k={k_exp} on {TOKEN_LEVEL}-level tokens, codesets {CODESETS_FILE} ({len(lab_e):,} patients). n (%) = patients whose dominant component is the bundle. "
        f"Entities ranked by {'NMF loading in the bundle' if RANK_BY == 'loading' else 'share of the bundle patients having the code'}; "
        "the percentage is the share of the bundle's patients with that code. The last two columns are left empty on purpose.").font = F(italic=True)
wb.save(XLSX_NMF)
print(f"wrote {os.path.abspath(XLSX_NMF)} ({k_exp} bundles)")
nmf_table


Read those out loud — several are recognisable clinical bundles. **That is already a crude
motif catalog**, from a three-line factorisation, no deep learning involved. Keep it in your
pocket for the conversation about effort versus value.

*In plain terms: no deep learning, no neural network -- just three lines of matrix math, and it already recovered groups that make clinical sense (a cardiac-workup bundle, a GERD-workup bundle, a procedural-sedation bundle). Worth remembering next time someone asks "why not jump straight to a fancy model?"*

### Reading the k=11 output

- 0 preventive, 1 acute infection pharmacy, 4 MSK/PT, 11 derm — background bundles, same
  reading as the breast-cancer cohort's k=12 fit on the same underlying claims population.
- 3 is procedural sedation (fentanyl/ondansetron/midazolam/propofol/dexamethasone) — a
  peri-procedural bundle, not itself diagnostic of psych vs cardiac.
- 5 is ambiguous, same as in the breast-cancer notebook: B12/vit D/free T4/ferritin/iron is
  a generic fatigue/anemia/thyroid panel, not obviously tied to either side of the
  hypothesis. Left unlabelled rather than forced into a bucket.
- 7 is acute respiratory infection (cough/URI/pharyngitis/COVID) — another benign-cause
  bundle, alongside GERD and costochondritis, that the cohort's working hypothesis says the
  data should be allowed to surface instead of psych.
- 8 is women's cancer screening (mammography/pap smear) and 10 is pregnancy/prenatal —
  both unrelated to chest pain, just base-rate utilization for this demographic.

**The cardiac side splits into two different kinds of evidence, not one bundle:**
- 2 is cardiac *workup* (troponin, EKG, CBC, metabolic panel) — the acute rule-out testing
  ordered *because of* the chest pain visit.
- 9 is cardiac *risk factors* (hypertension, hyperlipidemia, echocardiography, obesity,
  t2dm) — chronic comorbidity burden that predates and predisposes to a cardiac read,
  independent of any single visit.
- 6 is GERD/GI workup (endoscopy, biopsy, GERD, epigastric pain, colonoscopy) — the
  cohort's own stated alternative-explanation hypothesis, showing up as its own component.

**Notably absent: no component reads as psych/anxiety.** None of the 12 components put
anxiety, panic_attack, ssri, or benzodiazepine in their top-5 codes — the psych evidence
`chest_pain_labels()` uses to label patients doesn't dominate any single NMF bundle the way
cardiac workup, cardiac risk factors, and GERD workup each do. That's worth knowing before
Table 1: the NMF columns give a full, clinically-readable picture of *why* dx_9487 patients
were in the system, but "psych" isn't one of the picture's dominant themes at this k —
either it's spread thin across several components, or the corroborating evidence
(`chest_pain_labels()`'s `psych` side) is a weaker organizing signal in this data than
mastectomy/chemo evidence was for the breast-cancer cohort's fertility/oncology split.



### Where do the clinical codes sit? (loadings check)

Reading the top 5 codes per component can hide a code that matters but is never in anyone's top
5. This cell takes the clinical codes defined in `codesets.yaml` (the psych and cardiac evidence
tags that `chest_pain_labels()` uses, plus every other codeset) and shows, for each one, **how its
total loading is split across the components**. Each row sums to 100%.

*In plain terms: for every clinical code we defined by hand, which bundle "owns" it, and is it
spread thin across several bundles or concentrated in one? If the psych codes spread across many
components, that explains why no single bundle reads as psych.*

`owner` is the component with the largest share. `rank_in_owner` is the code's rank among all
codes inside that component (1 = the component's top code), so a high share with a rank in the
hundreds means the code is attached to that bundle but not what defines it. Only codes that have
a codeset tag appear here; conditions such as hypertension, hyperlipidemia or obesity are not in
`codesets.yaml`, so they are not covered by this check.

In [ ]:
K_CHECK = nmf.n_components_      # the last fit above
m8 = nmf if nmf.n_components_ == K_CHECK else NMF(n_components=K_CHECK, **NMF_KW).fit(Xs)
H8 = code_H(m8.components_)                       # crossed loadings summed over sex and band, so the columns are codes
# v3: columns of H8 are concept tokens; a codeset's loading is the SUM over the tokens it groups (a token counts in EVERY codeset it is in)
H8g = eo.group_loadings(H8, events)

# watch list comes from codesets.yaml (via eo), not hand-typed; the cohort anchor is excluded
WATCH = {
    'psych':          eo.PSYCH_TAGS,
    'cardiac':        eo.CARDIAC_TAGS,
    'other codesets': [t for t in eo.CODESETS
                       if t not in eo.PSYCH_TAGS + eo.CARDIAC_TAGS and t != 'chest_pain'],
}

rows, missing = [], []
for group, tags in WATCH.items():
    for tag in tags:
        if tag not in H8g.columns or H8g[tag].sum() == 0:
            missing.append(tag); continue
        share = H8g[tag] / H8g[tag].sum() * 100                       # this code's split across components
        own   = int(share.values.argmax())
        rows.append({'group': group, 'code': tag, **{f'c{i}': share.iloc[i] for i in range(K_CHECK)},
                     'owner': own, 'rank_in_owner': int((H8.iloc[own] > H8.iloc[own][H8.columns.intersection(list(eo.group_tokens(events, tag)))].max()).sum() + 1)})

loadings = pd.DataFrame(rows).set_index(['group', 'code'])
comp_cols = [f'c{i}' for i in range(K_CHECK)]
display(loadings.style.format({c: '{:.0f}' for c in comp_cols}).background_gradient(
        axis=1, cmap='Blues', subset=comp_cols))
print("components (n, top code):",
      {i: (int((m8.transform(Xs).argmax(1) == i).sum()), H8.iloc[i].idxmax()[:28]) for i in range(K_CHECK)})
print("codeset tags with no column (dropped by the prevalence floor, or never occur):", missing or "none")

In [ ]:
print(f"base fit: NMF k={nmf.n_components_}  (the `nmf` from the cell above)")
print(f"{N_BOOT_LOAD} refits")
cos, ari, _ = nmf_stability(nmf, Xs, n_boot=N_BOOT_LOAD)

dom_n = pd.Series(nmf.transform(Xs).argmax(1)).value_counts().reindex(range(nmf.n_components_), fill_value=0)
top3  = [', '.join(pd.Series(nmf.components_[c], index=nmf_cols)
                   .nlargest(3).index) for c in range(nmf.n_components_)]
stab = pd.DataFrame({'n': dom_n.values, 'top_codes': top3,
                     'cos_mean': cos.mean(0).round(3), 'cos_sd': cos.std(0).round(3),
                     'cos_min': cos.min(0).round(3)})
stab['verdict'] = np.select([stab.cos_mean >= .90, stab.cos_mean >= .75],
                            ['solid', 'suggestive'], 'unstable')
display(stab)
print(f"\nbootstrap ARI of argmax labels ({len(ari)} refits): mean {ari.mean():.3f}  sd {ari.std():.3f}  "
      f"(Day 2 cut-offs: >=0.75 solid, >=0.60 suggestive)")

plt.figure(figsize=(7, 3))
plt.errorbar(stab.index, stab.cos_mean, yerr=stab.cos_sd, fmt='o', capsize=4)
plt.axhline(0.90, ls='--', c='green',  label='solid')
plt.axhline(0.75, ls='--', c='orange', label='suggestive')
plt.xlabel('NMF component'); plt.ylabel('matched cosine vs base')
plt.xticks(stab.index); plt.legend(fontsize=8)
plt.title('NMF component stability under bootstrap refit')
plt.tight_layout(); plt.show()

In [ ]:
COHORT = 'chestpain'
# Persist the NMF components next to the .npz: n, top tokens with concept ids, a signature, and the names you give them.
# Part B of this notebook loads this file (COMP_JSON) for the component names. Workflow: run once -> copy the printed DESCR block
# (or paste names from Claude) into DESCR below -> run again. Components not listed in DESCR keep the names already saved
# in the file, as long as their top-5 tokens are unchanged.
import json, hashlib, os
CARD, RX, NONC, BG = 'cardiac workup', 'drug domain', 'non-cardiac explanation', 'background'
DESCR = {
    0: dict(name='outpatient cardiometabolic / preventive check-up (lipid panel, general exam, venipuncture, ECG, CMP, HbA1c, CBC, hypertension, periodic preventive visit, immunization)', group=CARD),
    1: dict(name="women's health and genitourinary (gyn exam, chlamydia / gonorrhea / HPV / trichomonas tests, cervical cytology, pregnancy test, urine culture, transvaginal echo)", group=BG),
    2: dict(name='acute-illness prescriptions (steroid packs, amoxicillin-clavulanate, azithromycin, cephalexin, NSAID, nasal steroid, hydrocodone-acetaminophen, muscle relaxant)', group=RX),
    3: dict(name='acute / ED workup (troponin, IV push injections, chest X-ray, lipase, ondansetron, ketorolac, BMP, CBC, prothrombin time)', group=CARD),
    4: dict(name='GI endoscopy and cancer screening (upper endoscopy and colonoscopy with biopsy, surgical pathology, GERD and other GI diagnoses, screening mammography, anesthesia, preventive visit)', group=NONC),
    5: dict(name='musculoskeletal / physical therapy (therapeutic exercise, manual therapy, chiropractic, PT evaluation; low back, neck, shoulder pain)', group=BG),
    6: dict(name='nutrition and endocrine labs (vitamin B12 / D, iron studies, folate, free T4, testosterone, general health panel, HbA1c)', group=BG),
    7: dict(name='acute respiratory infection (cough, URI, pharyngitis, sinusitis, COVID-19, strep / influenza antigen tests, injection)', group=NONC),
}
DESCR_K = len(DESCR)   # the k these names were written for (the Sean k = 8 fit, all components solid); with another k they are skipped

k_j       = nmf.n_components_
if DESCR and k_j != DESCR_K:
    print(f"k={k_j}, but the names in DESCR are for k={DESCR_K}: skipped"); DESCR = {}
NPZ_J     = f"nmf_k{k_j}_{TAG}.npz"
JSON_PATH = f"nmf_k{k_j}_{TAG}_components.json"
TOP_J     = 10

Hj     = code_H(nmf.components_)
lab_j  = nmf.transform(Xs).argmax(1)
meta_j = events.drop_duplicates('tag').set_index('tag')
old    = json.load(open(JSON_PATH))['components'] if os.path.exists(JSON_PATH) else {}

comps = {}
for c in range(k_j):
    row = Hj.iloc[c]
    sig = list(row.nlargest(5).index)
    toks = []
    for t, v in row.nlargest(TOP_J).items():
        if _is_group_token(t):                      # codeset tag: no single concept (helpers defined in the components-table cell)
            m = _top_member(t)
            toks.append(dict(token=t, concept_id=None,
                             concept_name=f"[codeset {t}] {int(_n_members[t])} concepts; most frequent: {m.concept_name}",
                             vocabulary_id='codeset', concept_code='', n_concepts=int(_n_members[t]),
                             loading=round(float(v), 6), share_pct=round(float(v / row.sum() * 100), 1)))
            continue
        m = meta_j.loc[t]
        toks.append(dict(token=t, concept_id=int(m.concept_id) if pd.notna(m.concept_id) else None,
                         concept_name=str(m.concept_name), vocabulary_id=str(m.vocabulary_id), concept_code=str(m.concept_code),
                         loading=round(float(v), 6), share_pct=round(float(v / row.sum() * 100), 1)))
    n    = int((lab_j == c).sum())
    keep = old.get(str(c), {})
    keep = keep if set(keep.get('signature', [])) == set(sig) else {}            # saved names only survive if the component is unchanged
    d    = DESCR.get(c, {})
    comps[str(c)] = dict(n=n, pct=round(n / len(lab_j) * 100, 2), signature=sig,
                         name=d.get('name', keep.get('name')), group=d.get('group', keep.get('group')),
                         status=d.get('status', keep.get('status')), top_tokens=toks)

out = dict(cohort=COHORT, k=int(k_j), npz=NPZ_J,
           npz_sha256=hashlib.sha256(open(NPZ_J, 'rb').read()).hexdigest() if os.path.exists(NPZ_J) else None,
           n_patients=int(len(lab_j)), components=comps)
with open(JSON_PATH, 'w') as f:
    json.dump(out, f, indent=2, ensure_ascii=False)
print(f"saved {JSON_PATH}  (k={k_j}, {len(lab_j):,} patients)\n")

print(f"{'comp':>4} {'n':>6} {'%':>5}  name  |  top tokens")
for c, d in comps.items():
    print(f"{c:>4} {d['n']:>6} {d['pct']:5.1f}  {d['name'] or '(unnamed)'}  |  " + ', '.join(t['token'].split('__')[0] for t in d['top_tokens'][:4]))
print("\n# paste-ready DESCR (edit name / group / status, then re-run this cell):")
print("DESCR = {")
for c, d in comps.items():
    extra = f", status={d['status']!r}" if d['status'] else ''
    print(f"    {c}: dict(name={d['name']!r}, group={d['group']!r}{extra}),")
print("}")


### Part A+ · What age and sex at exposure changed

Read in this order. All of it is descriptive: it shows what the crossed features did to the components, it does not rank them.

1. **Where each component's loading sits**: its share on female and male columns and on each age band (`'explicit'`: on the two risk flags), and the age-sex stratum (at the first chest pain) that dominates the component's patients.
2. **Is a component just an age-sex stratum?** `NMI` between the component of each patient and the patient's stratum at the first chest pain: 0 means the components say nothing about age and sex, 1 means they are the strata.
3. **Psych and cardiac groups**: how each group's loading (summed over sex and band) is split across components, and how common each group is by age band at the first chest pain.
4. **Same patients, baseline run**: how the components of this fit line up with the AHA k=8 components (`nmf_k8_chestpain_AHA.npz`).

*In plain terms: after splitting every kind of care by sex and age at the time of each event, do the bundles still look like kinds of care, or did they turn into "men 45-54", "women 25-34"? And does a bundle with anxiety in it show up now?*

*In plain terms for `'explicit'`: with only Sean's two age-sex flags added to the care codes, did any bundle turn into "men aged 45+" or "women aged 55+"? And do the bundles with anxiety, nitrates or cardiac testing show up?*

In [ ]:
from sklearn.metrics import normalized_mutual_info_score, adjusted_rand_score
k_ev   = nmf.n_components_
H_full = pd.DataFrame(nmf.components_, columns=nmf_cols)
lab_ev = nmf.transform(Xs).argmax(1)

TOK_SET = set(TOK_COLS)
S_SET   = set(BLOCKS['S'][1]) if CROSS else set()
is_x    = np.array([n in S_SET for n in nmf_cols])                      # the crossed columns of the matrix
if CROSS:               # share of each component's crossed loading on each sex and each age band (a crossed name is code_sex_band)
    Hx      = H_full.loc[:, is_x]
    _p      = pd.DataFrame([n.rsplit('_', 2) for n in Hx.columns], columns=['code', 'sex', 'band'], index=Hx.columns)
    by_sex  = Hx.T.groupby(_p['sex']).sum().T
    by_sex  = by_sex.div(by_sex.sum(axis=1), axis=0) * 100
    by_band = Hx.T.groupby(_p['band']).sum().T
    by_band = (by_band.div(by_band.sum(axis=1), axis=0) * 100).reindex(columns=eo.age_band_labels(AGE_EDGES)).dropna(axis=1, how='all')
if PLAIN and CROSS:     # 'both': how much of each component's loading sits on the plain columns
    plain_pct = H_full.loc[:, ~is_x].sum(axis=1) / H_full.sum(axis=1) * 100
if DEMO:                # 'extended': how much of each component's loading sits on the demographic columns
    D_SET    = set(BLOCKS[FLAGB][1])
    is_d     = np.array([n in D_SET for n in nmf_cols])
    demo_pct = H_full.loc[:, is_d].sum(axis=1) / H_full.sum(axis=1) * 100

strat = demo.reindex(person_ids)['stratum']
ok    = strat.notna().to_numpy()
rows  = []
for c in range(k_ev):
    m   = (lab_ev == c) & ok
    top = strat[m].value_counts(normalize=True)
    rows.append({'component': f'c{c}', 'n': int((lab_ev == c).sum()),
                 'top stratum at first chest pain': top.index[0] if len(top) else '', '% of its patients': round(100 * top.iloc[0], 1) if len(top) else np.nan,
                 **({f'crossed loading % {s}': round(by_sex.loc[c, s], 1) for s in by_sex.columns} if CROSS else {}),
                 **({'loading % on plain columns': round(plain_pct[c], 1)} if PLAIN and CROSS else {}),
                 **({'loading % on demographic columns': round(demo_pct[c], 1), 'top sex / age column': H_full.loc[c, is_d].idxmax()} if DEMO else {}),
                 'top columns': ', '.join(H_full.iloc[c].nlargest(4).index.str.slice(0, 34))})
display(pd.DataFrame(rows).set_index('component'))
if CROSS:
    print("share of each component's crossed loading by age band at the event (%):")
    display(by_band.round(0).astype(int).style.background_gradient(axis=1, cmap='Blues'))
nmi = normalized_mutual_info_score(strat[ok], lab_ev[ok])
print(f"NMI(component, age-sex stratum at the first chest pain) = {nmi:.3f}; the commonest stratum alone holds {100 * strat[ok].value_counts(normalize=True).iloc[0]:.1f}% of the patients")
if DEMO:
    print("'extended' / 'explicit': the demographic columns come from sex and age, so the NMI against the stratum is not independent evidence; see 'loading % on demographic columns' for how much of the fit they take.")

#### Where do the demographic columns sit? (loadings check, `'extended'`)

Part A+ above reads the loadings one component at a time: how much of a component's loading sits on sex and age. This reads them **one column at a time**, the way "Where do the clinical codes sit?" does for the codes: for each demographic column, how its total loading is split across the components. Each row sums to 100%.

`owner` is the component with the largest share. `rank_in_owner` is the column's rank among all the columns of that component (1 = the component's top column), so a high share with a rank in the hundreds means the column is attached to that component but does not define it. `patients with it %` is how many patients have the column on (the band, or the rule, met at any chest pain event): a common column gets some loading in several components for that reason alone.

The second table is the other direction, on the same base fit: each column's loading as a % of its component's **total** loading (codes included), and the sum over the demographic columns, which is the `loading % on demographic columns` of Part A+.

*In plain terms: which care pattern does "male", "age 35-44" or (in `'explicit'`) "man 45+" belong to, and is that pattern defined by sex or age, or is the column only along for the ride?*

In [ ]:
# the demographic columns of the fitted matrix: where does each one sit across the components?
if not DEMO:
    print(f"'{AGE_SEX_CROSS}': the matrix has no separate demographic columns (the crossed ones are read in Part A+), nothing to show here.")
else:
    D_cols = [n for n in nmf_cols if n in D_SET]                      # the demographic columns of the fitted matrix, in matrix order
    HD     = H_full[D_cols]                                           # component x column: the fitted loadings (the weight W_D is in them)
    prev_D = pd.Series(np.asarray((BLOCKS[FLAGB][0] > 0).mean(axis=0)).ravel() * 100, index=BLOCKS[FLAGB][1]).reindex(D_cols)
    label  = lambda n: n.replace('demo_sex_', 'sex ').replace('demo_age_', 'age ').replace('demo_', '').replace('_', ' ')

    rows, empty = [], []
    for col in D_cols:
        if HD[col].sum() == 0:
            empty.append(label(col)); continue
        share = HD[col] / HD[col].sum() * 100                         # this column's split across components
        own   = int(share.values.argmax())
        rows.append({'column': label(col), **{f'c{i}': share.iloc[i] for i in range(k_ev)}, 'owner': own,
                     'rank_in_owner': int((H_full.iloc[own] > H_full.iloc[own][col]).sum() + 1),
                     'patients with it %': prev_D[col]})
    sits      = pd.DataFrame(rows).set_index('column')
    comp_cols = [f'c{i}' for i in range(k_ev)]
    print("share of each column's loading that sits in each component (rows sum to 100):")
    display(sits.style.format({**{c: '{:.0f}' for c in comp_cols}, 'patients with it %': '{:.1f}'}).background_gradient(
            axis=1, cmap='Blues', subset=comp_cols))
    print(f"columns in the fitted matrix: {H_full.shape[1]:,}; patients per component:",
          {f'c{i}': int((lab_ev == i).sum()) for i in range(k_ev)})
    print("unstable in the stability table (cos_mean < 0.75):",
          [f'c{i}' for i in range(k_ev) if stab.verdict.iloc[i] == 'unstable'] or 'none')
    print("columns with no loading in any component:", empty or 'none')

In [ ]:
if DEMO:
    # the other direction, base fit: each component's loading on each column, as a % of that component's TOTAL loading (all columns, codes included)
    byc = HD.div(H_full.sum(axis=1), axis=0) * 100
    byc.columns = [label(n) for n in D_cols]
    byc.index   = [f'c{i}' for i in range(k_ev)]
    val_cols    = list(byc.columns)
    byc['all demographic columns'] = byc.sum(axis=1)                  # = 'loading % on demographic columns' of Part A+
    byc.insert(0, 'stability', stab.verdict.to_numpy())
    print("% of each component's total loading that sits on each demographic column:")
    display(byc.style.format({c: '{:.2f}' for c in byc.columns if c != 'stability'}).background_gradient(
            axis=None, cmap='Blues', subset=val_cols))

In [ ]:
# psych / cardiac / nitrate groups: loading per component, summed over the crossed columns (sex and band) of the group's codes
WATCH_A = ['anxiety', 'gerd', 'troponin', 'ekg', 'uw_cardiac_testing', 'uw_cad_dx', 'uw_abnormal_ekg', 'nitrates',
           'uw_gi_dx', 'uw_muscle_strain', 'costochondritis', 'uw_noncardiac_cp']
Hp    = eo.group_loadings(code_H(nmf.components_), events)             # codes -> groups; umbrella groups included
W     = Hp.reindex(columns=[g for g in WATCH_A if g in Hp.columns])
loadsplit = (W.div(W.sum(axis=0), axis=1) * 100).T.fillna(0).round(0).astype(int)
loadsplit.columns = [f'c{i}' for i in loadsplit.columns]
print("share of each group's loading that sits in each component (rows sum to 100):")
display(loadsplit.style.background_gradient(axis=1, cmap='Blues'))
print("groups without a column (dropped by the prevalence floor, or never occur):", [g for g in WATCH_A if g not in W.columns] or "none")

# how common each group is by age band (% of the patients in the band with at least one event of the group), plain codes
_gi = [g for g in WATCH_A if g in TOK_COLS]
_pm = pd.DataFrame(Xs_tok[:, [TOK_COLS.index(g) for g in _gi]].toarray(), columns=_gi, index=person_ids)
_ab = demo.reindex(person_ids)['age_band']
print("\n% of patients with the group, by age band at the first chest pain:")
display((_pm.groupby(_ab.to_numpy()).mean() * 100).reindex(eo.age_band_labels(AGE_EDGES)).round(1))

In [ ]:
# Where Sean's loadings sit: each component's loading on Sean's code sets, as a % of the component's TOTAL loading (all plain codes).
# The cells above read it code set by code set (how one code set splits across components); this reads it component by component.
H_s    = code_H(nmf.components_)                                   # components x plain tokens: the demographic columns of 'extended' / 'explicit' are not codes, they drop out here
Hg_s   = eo.group_loadings(H_s, events)                            # components x code sets: a code set's loading is the sum over its tokens (a token counts in every code set it is in)
tot_s  = H_s.sum(axis=1)
pct_s  = Hg_s.div(tot_s, axis=0) * 100
no_col = [t for t in codesets if t not in pct_s.columns or pct_s[t].sum() == 0]          # no token of its own in the fitted matrix: dropped by the floor, never occurs, or its concepts belong to an earlier code set
pct_s  = pct_s[[t for t in codesets if t not in no_col]]
sean_tok   = set(events.loc[events['in_codeset'], 'tag'].unique()) & set(H_s.columns)       # every token of any code set, once
sean_share = H_s[sorted(sean_tok)].sum(axis=1) / tot_s * 100
k_s    = H_s.shape[0]
comps  = [f'c{i}' for i in range(k_s)]

tab = pct_s.round(1)
tab.index = comps
tab.insert(0, 'all Sean code sets %', sean_share.round(1).to_numpy())
if 'stab' in globals() and len(stab) == k_s:
    tab.insert(0, 'stability', stab['verdict'].to_numpy())
tab['top Sean code set'] = pct_s.idxmax(axis=1).where(pct_s.max(axis=1) > 0, '').to_numpy()       # empty when the component has no Sean loading at all
tab['top 3 columns of the component'] = [', '.join(H_s.iloc[i].nlargest(3).index.str.slice(0, 26)) for i in range(k_s)]
num = ['all Sean code sets %'] + list(pct_s.columns)
print("% of each component's total loading that sits on Sean's code sets (a row adds up to less than 100: the rest is untagged concepts):")
display(tab.style.format({c: '{:.1f}' for c in num}).background_gradient(axis=None, cmap='Blues', subset=list(pct_s.columns)))
print("code sets with no loading in any component:", no_col or 'none')

fig, ax = plt.subplots(1, 2, figsize=(14, 0.5 * k_s + 3.2), gridspec_kw={'width_ratios': [3, 1]})
A = pct_s.to_numpy()
ax[0].imshow(A, aspect='auto', cmap='Blues')
ax[0].set_xticks(range(A.shape[1])); ax[0].set_xticklabels(pct_s.columns, rotation=60, ha='right', fontsize=8)
ax[0].set_yticks(range(k_s)); ax[0].set_yticklabels(comps)
for i in range(k_s):
    for j in range(A.shape[1]):
        if A[i, j] >= 0.5:
            ax[0].text(j, i, f'{A[i, j]:.1f}', ha='center', va='center', fontsize=7, color='white' if A[i, j] > 0.6 * A.max() else 'black')
ax[0].set_title("% of each component's loading on Sean's code sets", fontsize=10)
ax[1].barh(range(k_s), sean_share, color='#2C5F6F', label="Sean's code sets")
ax[1].barh(range(k_s), 100 - sean_share, left=sean_share, color='#B8C4C9', label='untagged concepts')
for i, v in enumerate(sean_share):
    ax[1].text(v + 1, i, f'{v:.0f}%', va='center', fontsize=8)
ax[1].set_ylim(k_s - 0.5, -0.5); ax[1].set_yticks([]); ax[1].set_xlim(0, 100)
ax[1].set_xlabel('% of the component\'s total loading'); ax[1].legend(fontsize=8, loc='lower right')
ax[1].set_title("Sean's code sets vs the rest", fontsize=10)
plt.tight_layout(); plt.show()


In [ ]:
BASE_NPZ = 'nmf_k8_chestpain_AHA.npz'
if os.path.exists(BASE_NPZ):
    _z    = np.load(BASE_NPZ)
    _base = pd.Series(_z['labels'], index=_z['pids'])
    _both = _base.index.intersection(pd.Index(person_ids))
    _new  = pd.Series(lab_ev, index=person_ids).loc[_both]
    print(f"patients in both runs: {len(_both):,}   ARI(this k={k_ev} vs AHA k={_base.nunique()}) = {adjusted_rand_score(_base.loc[_both], _new):.3f}")
    display(pd.crosstab(_base.loc[_both].rename('AHA k=8 component'), _new.rename('this run')).style.background_gradient(axis=1, cmap='Blues'))
else:
    print(f"{BASE_NPZ} not found: run the AHA notebook first to compare against it")

# Part B · Day 3 — sequence motifs inside the NMF bundles

In [ ]:
seqs = eo.sequences(events, col='tag', dedup_same_day=True, collapse_repeats=True)
pids = sorted(seqs)
db   = [seqs[p] for p in pids]

print("example timeline:")
print(' -> '.join(db[1][:12]))
print("\nmedian sequence length:", int(np.median([len(s) for s in db])))

### The preprocessing decisions, restated

The first two live in `eo.sequences()` and both are arbitrary:

1. **Same-day ties.** Claims carry a date, not a timestamp, so `DOMAIN_ORDER` breaks ties.
   Visit, condition, procedure, measurement, drug, observation.
2. **Refill collapsing.** Consecutive repeats of the same tag become one token, otherwise
   every pattern is `metformin -> metformin -> metformin`. Chronicity is real signal, so
   capture it as a separate feature — fill count, days supply — not as sequence position.
3. **Visits are not in the sequence at all (v2).** They were the most common tokens in v1 and
   made every motif list a chain of pharmacy / office / ED visits. Dropping them removes that
   care-setting bias; it also removes the `ed_visit` tag, so any motif that depended on "went to the
   ED" is invisible here by construction.

## 1 · PrefixSpan, implemented


In [ ]:
import collections

def prefixspan(db, min_sup, max_len=4):
    """Minimal PrefixSpan over single-item sequences.

    db      : list of lists of tokens
    min_sup : minimum number of sequences containing the pattern
    returns : DataFrame of (pattern, support, support_pct)
    """
    n, results = len(db), []

    def scan(projected, prefix):
        counts = collections.Counter()
        for seq in projected:
            for item in sorted(set(seq)):                   # count each sequence once
                counts[item] += 1
        for item, c in counts.items():
            if c < min_sup:
                continue
            new_prefix = prefix + [item]
            results.append((tuple(new_prefix), c))
            if len(new_prefix) >= max_len:
                continue
            # project: keep the suffix after the FIRST occurrence of item
            nxt = [seq[seq.index(item) + 1:] for seq in projected if item in seq]
            if nxt:
                scan(nxt, new_prefix)

    scan([list(s) for s in db], [])
    return (pd.DataFrame(results, columns=['pattern', 'support'])
              .assign(support_pct=lambda d: (d.support / n * 100).round(1)))

## 2 · Discriminative mining — motifs per NMF component

Raw support finds whatever is common, which is mostly ambient care. What you want is patterns
common **inside** a group and rare **outside** it. Here the groups are the **NMF components of
`Day1_nmf_v3`** (concept-level tokens, visits pruned; `K` is set in the NMF cell below).

What the v2 loadings showed, so the motifs below can be read against it:

- **c3 acute / ED workup** owns `troponin` (62%, rank 1); **c0 outpatient cardiometabolic workup** owns
  `ekg`, `stress_test`, `hba1c`, `t2dm`. `cad` and `coronary_angio` are split between the two.
- **No anxiety / panic bundle exists**: `anxiety` is spread 10-22% over c0, c2, c3, c7;
  `panic_attack` sits in c3 (28%) and c7 (27%). `ssri` / `benzodiazepine` load on the prescription
  bundle c1, but so do `beta_blocker` and `statin`.
- **c4** is the GI / screening bundle that owns `gerd`.

Day 3 asks what order those codes come in, per component.


In [ ]:
# Part B starts from the NMF fitted in Part A (same events, same prevalence floor, same anchor drop): no refit, no cache file.
K    = nmf.n_components_
H_full = pd.DataFrame(nmf.components_, columns=nmf_cols)   # the columns the NMF was fitted on (Part A+ reads this one)
H    = code_H(nmf.components_)                             # Part B names codes: crossed loadings summed over sex and band
lab  = pd.Series(nmf.transform(Xs).argmax(1), index=person_ids)
comp = lab.reindex(pids).fillna(-1).astype(int).to_numpy()   # aligned with db / pids

CARD, RX, NONC, BG = 'cardiac workup', 'drug domain', 'non-cardiac explanation', 'background'
COMP = {}                                                     # auto-names from the top tokens; real names come from the components JSON below
for c in range(K):
    top = H.iloc[c].idxmax()
    COMP[c] = (top[:20], f'c{c}: ' + ', '.join(H.iloc[c].nlargest(3).index.str.split('__').str[0]), BG)
# Names from the components JSON written by the last cell of Part A: override COMP for every component whose top-5 tokens still match
# (>= 4 of 5); otherwise the auto-name is kept and a warning is printed. Point COMP_JSON at another file to reuse it.
import json
COMP_JSON = f"nmf_k{K}_{TAG}_components.json"
if os.path.exists(COMP_JSON):
    J = json.load(open(COMP_JSON))['components']
    for c in range(K):
        j = J.get(str(c))
        if not j or not j.get('name'):
            continue
        if len(set(j['signature']) & set(H.iloc[c].nlargest(5).index)) < 4:
            print(f"WARNING c{c}: top tokens in {COMP_JSON} differ from this NMF, its name is ignored")
            continue
        COMP[c] = (COMP[c][0], j['name'], j.get('group') or COMP[c][2]) + ((j.get('status') or COMP[c][3],) if len(COMP[c]) > 3 else ())
    print(f"component names loaded from {COMP_JSON}")
else:
    print(f"{COMP_JSON} not found: components are auto-named (name them in DESCR in the last cell of Part A)")
print("top tokens per component:")
for c in range(K):
    print(f"  c{c}:", ', '.join(H.iloc[c].nlargest(4).index.str.slice(0, 32)))

print(f"{'comp':>4} {'n':>6} {'%':>5}  {'group':<24} name")
for c, v in COMP.items():
    n = int((comp == c).sum())
    print(f"{c:>4} {n:>6} {n / len(comp) * 100:5.1f}  {v[2]:<24} {v[1]}")


In [ ]:
from joblib import Parallel, delayed

def contains(seq, pat):
    it = iter(seq)
    return all(x in it for x in pat)

def discriminative(mask, candidates, sequences_db=None, min_in=0.12):
    sdb = sequences_db if sequences_db is not None else db
    out = []
    for p in candidates:
        h = np.array([contains(s, p) for s in sdb])
        si, so = h[mask].mean(), h[~mask].mean()
        if si >= min_in:
            out.append({'pattern': ' -> '.join(p), 'sup_in': round(si, 3),
                        'sup_out': round(so, 3), 'lift': round((si + .01) / (so + .01), 1)})
    if not out:
        return pd.DataFrame(columns=['pattern', 'sup_in', 'sup_out', 'lift'])
    return pd.DataFrame(out).sort_values(['lift', 'pattern'], ascending=[False, True]).reset_index(drop=True)

def _component_job(sdb_in, sdb_out, rel_sup, max_cand):
    """One component: LOCAL mining on its own patients, then lift vs the out-sample.
    Takes only its own slices, so it ships cheaply to a worker process."""
    pats_c = prefixspan(sdb_in, max(3, int(rel_sup * len(sdb_in))), max_len=4)
    pats_c = pats_c[pats_c.pattern.map(len) >= 3].sort_values(['support', 'pattern'], ascending=[False, True]).head(max_cand)
    sdb    = sdb_in + sdb_out
    mask   = np.arange(len(sdb)) < len(sdb_in)
    return discriminative(mask, [tuple(p) for p in pats_c.pattern], sequences_db=sdb, min_in=rel_sup)

def component_motifs(rel_sup=0.10, max_cand=60, n_out=1500, max_in=3000, seed=0, n_jobs=None):
    """All components in parallel (they are independent); largest first so no big job starts last.
    Each component is mined on a random sample of at most `max_in` of its patients (PrefixSpan is
    pure Python; a 30,000-patient component would take hours). Raise max_in once you know the runtime."""
    n_jobs = n_jobs or min(len(COMP), max(1, (os.cpu_count() or 2) - 2))   # leave 2 cores free
    order  = sorted(COMP, key=lambda c: -int((comp == c).sum()))
    jobs   = []
    for c in order:
        idx_in  = np.flatnonzero(comp == c)
        idx_in  = np.random.default_rng(seed).choice(idx_in, min(max_in, len(idx_in)), replace=False)
        idx_out = np.flatnonzero(comp != c)
        idx_out = np.random.default_rng(seed).choice(idx_out, min(n_out, len(idx_out)), replace=False)
        jobs.append(delayed(_component_job)([db[i] for i in idx_in], [db[i] for i in idx_out],
                                            rel_sup, max_cand))
    return dict(zip(order, Parallel(n_jobs=n_jobs, verbose=5)(jobs)))

# the local mining is the point: a global 4% floor would bury the 6% components entirely
motifs = component_motifs()
for c, (_, name, role) in COMP.items():
    d = motifs[c]
    print(f"\n===== component {c} [{role}]: {name}  (n={(comp == c).sum()}, mined on {min(3000, (comp == c).sum())})")
    if len(d):
        display(d.head(4))
    else:
        print("  no ordered pattern of length >= 3 reaches the in-component support floor")


In [ ]:
# Held-out pass rate, family clustering and same-day flag (the table below reads them). Computed in motif_robustness/hybrid.py, nothing is shown here.
import sys, time
sys.path.insert(0, "../repr_learn_letrozole")          # shared helpers, not copied
from motif_robustness.preprocess import build_patient_seqs
from motif_robustness.hybrid import add_heldout_chestpain

REL_SUP        = 0.10                     # component_motifs() default support floor
HO_SEEDS       = tuple(range(101, 121))   # 20 splits; both directions are scored -> 40 runs
HO_MIN_OR      = 2.0                      # lower 95% bound of the odds ratio on the held-out half
HO_MIN_RATE_LO = 0.70                     # family pass rate (lower Wilson bound) needed to call a motif stable
FAM_DIST       = 0.2
SAME_DAY_RATIO = 0.5                      # strict-date support / tie-ordered support below this = same-day bundle

_p, _ps = build_patient_seqs(events)      # same dedup / collapse rules as eo.sequences, but keeps the dates
assert _p == pids and all(_ps[p].tokens == s for p, s in zip(pids, db)), "dated sequences differ from db"
patseqs = [_ps[p] for p in pids]
n_in = {c: int((comp == c).sum()) for c in COMP}

t0 = time.perf_counter()
hyb, ho_runs = add_heldout_chestpain(motifs, comp, patseqs, rel_sup=REL_SUP, ho_seeds=HO_SEEDS, ho_min_or=HO_MIN_OR,
                                     ho_min_rate_lo=HO_MIN_RATE_LO, fam_dist=FAM_DIST, same_day_ratio=SAME_DAY_RATIO)
print(f"held-out pass rate: {(time.perf_counter() - t0):.0f} s")


Each component now has **candidate motifs attached** — an ordered sequence, not just a set of
prevalent codes. Things to check:

- Does **c3** read as a workup path (troponin, EKG, imaging, angiography, in that order) and does
  **c0** read as a slower outpatient path (EKG, stress test, labs)?
- Do `anxiety` / `panic_attack` show up as a motif *before* troponin in **c3**, or only after it?
  Without visits, a "panic in the ED" story has to be carried by the workup codes alone.
- Do the background bundles (c2, c5, c6, c7) stay generic?

### A second lens: the Day 1 psych / cardiac probe

`chest_pain_labels()` is a direct rule-based split built from corroborating co-evidence. It is a
probe, not the structure: the labels come from `PSYCH_TAGS` / `CARDIAC_TAGS`, so a motif made of
those same tags partly rediscovers the label rule. Mined on a random 5,000-patient sample, since
every candidate is scanned against every sequence.


In [ ]:
weak = eo.chest_pain_labels(events)
wl   = weak.set_index('person_id').chest_pain_label

# All patients here have chest_pain by cohort construction, so no filtering needed. LOCAL mining on a
# fixed random sample: candidates x sequences scans are the cost on a cohort this size.
rng   = np.random.default_rng(0)
samp  = rng.choice(len(db), min(5_000, len(db)), replace=False)
db_s  = [db[i] for i in samp]
y_s   = np.array([wl.get(pids[i], 'unlabelled') for i in samp])
print("sampled patients:", len(db_s),
      "  labelled:", int((y_s != 'unlabelled').sum()),
      "  ", dict(zip(*np.unique(y_s, return_counts=True))))

pats_cp = prefixspan(db_s, max(3, int(0.08 * len(db_s))), max_len=4)
C_cp = [tuple(p) for p in pats_cp[pats_cp.pattern.map(len) >= 3]
        .sort_values(['support', 'pattern'], ascending=[False, True]).head(60).pattern]

for g in ['psych', 'cardiac']:
    print(f"\n=== {g} evidence")
    display(discriminative(y_s == g, C_cp, sequences_db=db_s, min_in=0.10).head(6))


**This is the deliverable.** Ordered patterns, with in-group and out-group support, that read
as sentences. No transformer, no embeddings, no non-commercial licence to negotiate.

## 3 · Interval relations — when order is not enough

Order says "A before B." It does not say "A the same week" versus "A two years before B,"
which is the actual dx_9487 discriminator: is the anxiety history old and unrelated, or did
a cardiac workup follow within days. Allen's interval algebra defines 13 relations; for
point events on dates, a **signed gap** carries most of the value.

### 3b · Does psych evidence come before or after troponin?

Motif mining only surfaces patterns above a support floor, and psychiatric care is a minority inside
every bundle, so it can miss exactly this question. This asks it directly, with no floor: for each
patient with both, the days from their first psych evidence to their first troponin.

- **Psych evidence** = an anxiety or panic diagnosis, or a psychiatric evaluation / psychotherapy code
  (`psych_eval_therapy`). SSRIs and benzodiazepines are left out: they are prescribed for many reasons.
- Positive = psych evidence **first**, then troponin. Negative = troponin first.
- Patients with psych evidence and no troponin are not in this comparison (no workup in the data).
- Only meaningful where history covers the earlier event, as in section 3.


In [ ]:
PSYCH = ['anxiety']
ev    = events[['person_id', 'event_date', 'concept_id', 'groups']]                  # no copy of the full frame
first = lambda tags: ev.loc[eo.group_mask(ev, tags)].groupby('person_id').event_date.min()

both = pd.concat([first(PSYCH).rename('psych'), first(['troponin']).rename('trop')], axis=1).dropna()
both['days_psych_to_trop'] = (both.trop - both.psych).dt.days
both['bundle'] = pd.Series(comp, index=pids).reindex(both.index)
both = both[both.bundle >= 0]                                   # drop patients with no assigned bundle

def timing(d):
    g = d.days_psych_to_trop
    return pd.Series({'n': len(d), 'psych first %': round((g > 0).mean() * 100, 1),
                      'same day %': round((g == 0).mean() * 100, 1),
                      'troponin first %': round((g < 0).mean() * 100, 1),
                      'median days': g.median()})

tab = both.groupby('bundle').apply(timing)
tab.index = [f"c{int(b)} {COMP[int(b)][1][:34]}" for b in tab.index]
tab.loc['ALL (psych + troponin)'] = timing(both)
display(tab.sort_values('n', ascending=False))

c3d = both[both.bundle == 3].days_psych_to_trop
plt.figure(figsize=(8.5, 3))
plt.hist(both.days_psych_to_trop.clip(-1500, 1500), bins=60, alpha=.5, density=True, label=f"all (n={len(both):,})")
plt.hist(c3d.clip(-1500, 1500), bins=60, alpha=.5, density=True, label=f"c3 {COMP[3][1][:30]} (n={len(c3d):,})")
plt.axvline(0, c='k', lw=1); plt.yticks([])
plt.xlabel('days from first psych evidence to first troponin   (negative = troponin first; clipped at +-1500)')
plt.legend(fontsize=8); plt.title('psych evidence vs troponin timing')
plt.tight_layout(); plt.show()


## 4 · Trajectory graph — the readable artifact

Aggregate consecutive transitions into a weighted directed graph. This is the closest thing
to a chest-pain triage flowchart, but derived from observed transitions rather than a
guideline document — Ethos has no chest-pain ruleset to start from, so the agreed method is
to start from the data and check clinical plausibility afterwards.

In [ ]:
import networkx as nx

FOCUS = {'chest_pain', 'ekg', 'troponin', 'uw_cardiac_testing', 'uw_cad_dx', 'uw_abnormal_ekg', 'nitrates',
         'anxiety', 'gerd', 'uw_gi_dx', 'uw_muscle_strain', 'costochondritis', 'uw_noncardiac_cp'}

# the graph is about curated clinical steps: relabel each concept token by its group, drop uncurated ones, collapse repeats
import itertools
_tg = events.drop_duplicates('tag').dropna(subset=['groups']).set_index('tag')['groups'].astype(str).str.split('|').to_dict()   # token -> ALL its groups
gseqs = {p: [g for g, _ in itertools.groupby(g for t in seqs[p] for g in _tg.get(t, ()))] for p in pids}
db_cp = [[c for c in gseqs[p] if c in FOCUS] for p in pids if 'chest_pain' in gseqs[p]]
print(f"{len(db_cp)} chest pain patients")

cnt = collections.Counter()
for s in db_cp:
    for u, v in zip(s, s[1:]):
        cnt[(u, v)] += 1
G = nx.DiGraph()
for (u, v), w in cnt.items():
    if w >= max(5, int(0.02 * len(db_cp))):
        G.add_edge(u, v, weight=w)
print(G.number_of_nodes(), "nodes,", G.number_of_edges(), "edges")


### Trajectory graph on a time axis
x = median days from the first chest pain to the first time the patient has that group (negative = before it), y = the component group that owns the tag's NMF loading. Nodes in a lane are staggered by x, so labels cannot overlap. The cell above only builds the transitions (`G`).

In [ ]:
# Trajectory graph on a time axis: x = median days from the patient's first chest pain to the first time the patient has that group
# (log-compressed, 0 = chest pain), y = the component group that owns the tag's NMF loading. Nodes in one lane are staggered by x, so labels cannot overlap.
# Edges are the transitions in G (cell above). Needs the graph cell above and the dated sequences (patseqs) from the hybrid section.
ANCHOR      = 'chest_pain'
ANCHOR_NAME = ANCHOR.replace('_', ' ')
MIN_N       = 20                                                   # fewer patients with both -> no timing, node parked at the right edge
SLOT, LABEL_IN = 1.0, 1.15                                          # vertical stagger step (y units), room one label needs on the x axis (inches)

_nodes  = set(G.nodes())
_have   = [i for i, p in enumerate(pids) if ANCHOR in gseqs[p]]     # the cohort of the graph
_first  = {n: [] for n in _nodes}
_a_tok  = {t for t, gs in _tg.items() if ANCHOR in gs}
for i in _have:
    ps = patseqs[i]
    a0 = min((ps.dpos[t][0] for t in _a_tok if t in ps.dpos), default=None)
    if a0 is None:
        continue
    best = {}
    for t, dl in ps.dpos.items():
        for g in _tg.get(t, ()):
            if g in _nodes and (g not in best or dl[0] < best[g]):
                best[g] = dl[0]
    for g, d in best.items():
        _first[g].append(d - a0)
med = {g: float(np.median(v)) for g, v in _first.items() if len(v) >= MIN_N}
med[ANCHOR] = 0.0

Hg_    = Hg if 'Hg' in globals() else eo.group_loadings(H, events)
comps_ = list(dict.fromkeys(v[2] for v in COMP.values()))           # component groups, in COMP order
def owner_lane(g):
    if g == ANCHOR:
        return 'anchor'
    if g not in Hg_.columns or Hg_[g].sum() == 0:
        return 'not in NMF'
    share = {l: Hg_.loc[[k for k, v in COMP.items() if v[2] == l], g].sum() / Hg_[g].sum() for l in comps_}
    return max(share, key=share.get)
lane_of  = {n: owner_lane(n) for n in _nodes}
LANES    = [l for l in comps_ + ['anchor', 'not in NMF'] if l in set(lane_of.values())]
_pal     = ['#2f855a', '#c05621', '#718096', '#2b6cb0', '#b83280', '#b7791f']
LANE_COL = {**{l: _pal[i % len(_pal)] for i, l in enumerate(comps_)}, 'anchor': '#d69e2e', 'not in NMF': '#a0aec0', **globals().get('GCOL', {})}

tf = lambda d: np.sign(d) * np.log1p(np.abs(d) / 30)                # days -> compressed axis, 0 stays 0
xs = {g: tf(m) for g, m in med.items() if g in _nodes}
x_park = (max(xs.values()) if xs else 0) + 1.4
_all_x = list(xs.values()) + ([x_park] if any(n not in xs for n in _nodes) else [])
MIN_DX = LABEL_IN * (max(_all_x) - min(_all_x) + 1) / 12.0           # x units that fit one label on a 14 in wide figure
order  = [0] + [s * k for k in range(1, 13) for s in (1, -1)]
slot_of, lane_k = {}, {}
for l in LANES:
    members = sorted((n for n in _nodes if lane_of[n] == l), key=lambda n: (xs.get(n, x_park), n))
    last = {}                                                      # slot -> x of the last node placed there
    for n in members:
        x = xs.get(n, x_park)
        k = next(k for k in order if x - last.get(k, -1e9) >= MIN_DX)
        last[k] = x; slot_of[n] = (l, k, x)
    lane_k[l] = max((abs(k) for (ll, k, _) in slot_of.values() if ll == l), default=0)
heights = {l: 2 * lane_k[l] * SLOT + 1.8 for l in LANES}            # a lane is as tall as its stagger needs
centers, y_top = {}, 0.0
for l in LANES:
    centers[l] = y_top - heights[l] / 2; y_top -= heights[l]
pos_t = {n: (x, centers[l] + k * SLOT) for n, (l, k, x) in slot_of.items()}

fig, ax = plt.subplots(figsize=(14, max(5, 0.6 * (-y_top) + 1.8)))
for l in LANES:
    ax.axhspan(centers[l] - heights[l] / 2, centers[l] + heights[l] / 2, color=LANE_COL.get(l, '#e2e8f0'), alpha=.08, lw=0)
    ax.text(-0.01, centers[l], l, transform=ax.get_yaxis_transform(), ha='right', va='center', fontsize=9, fontweight='bold')
w = np.array([G[u][v]['weight'] for u, v in G.edges()], dtype=float)
nx.draw_networkx_edges(G, pos_t, ax=ax, width=1 + 4 * w / w.max(), alpha=.3, arrowsize=14, node_size=700, connectionstyle='arc3,rad=0.15')
nx.draw_networkx_nodes(G, pos_t, ax=ax, node_size=700, node_color=[LANE_COL.get(lane_of[n], '#e2e8f0') for n in G.nodes()], edgecolors='#4a5568', linewidths=.8)
nx.draw_networkx_labels(G, {n: (x, y - 0.42) for n, (x, y) in pos_t.items()}, ax=ax, font_size=8,
                        bbox=dict(boxstyle='round,pad=0.12', fc='white', ec='none', alpha=.85))
ax.tick_params(axis='x', bottom=True, labelbottom=True)              # networkx hides the ticks
tick_d = [-365, -90, -30, 0, 30, 90, 365, 730]
ax.set_xticks([tf(d) for d in tick_d]); ax.set_xticklabels([f"{d:+d} d" if d else ANCHOR_NAME for d in tick_d], fontsize=8)
ax.axvline(0, color='#d69e2e', lw=1, ls='--', alpha=.7)
if any(n not in xs for n in _nodes):
    ax.text(x_park, 0.3, f'timing n<{MIN_N}', ha='center', fontsize=8, color='#718096')
ax.set_yticks([]); ax.spines[['left', 'right', 'top']].set_visible(False)
ax.set_xlabel(f"median days from first {ANCHOR_NAME} to first occurrence of the group (log-compressed)")
ax.set_title(f'{ANCHOR_NAME} trajectory graph on a time axis -- lanes = owning component group')
plt.margins(x=.06); plt.tight_layout(); plt.show()
print({l: int(sum(1 for n in _nodes if lane_of[n] == l)) for l in LANES}, "nodes per lane;", sum(1 for n in _nodes if n not in xs), "without timing")


**How to read it.** Left to right is time: each tag sits at the median number of days between the patient's first chest pain and the first time they have that tag (log-compressed; left = before chest pain, right = after, the dashed line is chest pain). Each row (lane) is the NMF component role that owns most of the tag's loading (cardiac workup, drug domain, non-cardiac explanation or background, as set in `COMP`). `chest_pain` has its own lane because it is the cohort anchor and not in the NMF; a tag too rare to reach the NMF goes to a "not in NMF" lane. Tags inside a lane are staggered up and down only so the labels do not overlap: the vertical offset has no meaning. A tag with fewer than 20 patients who have both it and chest pain has no timing and is parked at the right edge. Positions use the *first* occurrence, so a tag that repeats (ECG, statin) can look earlier than the episode next to chest pain. A lane is a loading share, not a diagnosis. Arrows are the observed transitions between consecutive curated steps; thicker = more patients.

Two arms should be visible out of `chest_pain`: one through troponin, stress test,
angiography and statins; the other through anxiety, panic_attack, ssri and benzodiazepine.
Whichever arm dominates numerically is the answer to the working hypothesis — let the graph
say which, rather than assuming.

## Carry forward

A motif catalog for dx_9487 triage, obtained without a foundation model.

## 5 · Member table — which concepts sit behind each token
One row per OMOP concept with the token it carries (`token_members_k{K}_{TAG}.csv`). At group level a token is a codeset tag with many concepts; an engine entity or predicate is then `concept_id IN (...)` over its rows. Includes codeset members never seen in this cohort (`in_cohort` False). `groups` lists every tag a concept belongs to; the token is the first one in YAML order. Run before the final table (needs K, H, hyb); the final table shows the members of its group tokens.

In [ ]:
COHORT = 'chestpain'
# Member table: one row per OMOP concept with the token it carries, so a predicate / engine entity can be written as `concept_id IN (...)`
# instead of re-resolving the yaml later. At TOKEN_LEVEL='group' a token is a codeset tag and has many rows; at 'concept' it has one.
# Includes (a) every concept seen in the cohort's events (n_events > 0) and (b) every codeset member never seen in the cohort (n_events = 0,
# in_cohort False), because an entity defined on this cohort must also cover codes that happen to be absent from it.
# Needs Part B (K, H, hyb). Run it before the final-table cell, which reads `member_tbl` for its Members columns. File = token_members_k{K}_{TAG}.csv
import collections
MEMBERS_CSV = f"token_members_k{K}_{TAG}.csv"
META  = [c for c in ('concept_name', 'vocabulary_id', 'concept_code', 'concept_class_id', 'domain') if c in events.columns]

first = events.drop_duplicates('concept_id').set_index('concept_id')
cnt   = events.groupby('concept_id').person_id.agg(n_events='size', n_patients='nunique')
seen  = first[['tag'] + META].rename(columns={'tag': 'token'}).join(cnt)
seen['groups'] = first['groups'].astype(str).where(first['groups'].notna(), '')            # every codeset the concept is in, '|'-joined

lookup, members = {}, {}                                                                   # same rule as eo.tag_events: first tag in yaml order wins
for tag, ids in codesets.items():
    for cid in ids:
        lookup.setdefault(cid, tag); members.setdefault(cid, []).append(tag)
cmeta  = concept.drop_duplicates('concept_id').set_index('concept_id')
unseen = cmeta.reindex([c for c in members if c not in seen.index]).reindex(columns=META)
unseen['token']    = unseen.index.map(lookup) if TOKEN_LEVEL == 'group' else ''            # at concept level the token of an unseen concept does not exist
unseen['groups']   = unseen.index.map(lambda c: '|'.join(members[c]))
unseen['n_events'] = 0
unseen['n_patients'] = 0

member_tbl = pd.concat([seen, unseen]).rename_axis('concept_id').reset_index()
assert member_tbl.concept_id.is_unique, "a concept carries more than one token"
member_tbl['in_cohort']        = member_tbl.n_events > 0
member_tbl['token_n_concepts'] = member_tbl.groupby('token').concept_id.transform('size').where(member_tbl.token != '')      # all members, seen or not
member_tbl['token_n_patients'] = member_tbl.token.map(events.groupby('tag').person_id.nunique()).astype('Int64')     # patients with ANY member of the token
member_tbl['in_nmf']           = member_tbl.token.isin(H.columns)
member_tbl['nmf_owner']        = member_tbl.token.map(H.idxmax(axis=0)).astype('Int64')                              # component with the largest loading (blank = not in the NMF)
member_tbl['owner_share']      = member_tbl.token.map(H.max(axis=0) / H.sum(axis=0)).round(3)
_in_motifs = collections.Counter(t for d in hyb.values() if len(d) for p in d.pattern for t in set(p.split(' -> ')))
member_tbl['n_motifs']         = member_tbl.token.map(lambda t: _in_motifs.get(t, 0))
member_tbl = member_tbl.sort_values(['token_n_patients', 'token', 'n_patients'], ascending=[False, True, False], na_position='last')
member_tbl = member_tbl[['token', 'concept_id'] + META + ['n_events', 'n_patients', 'groups', 'in_cohort', 'token_n_concepts', 'token_n_patients', 'in_nmf', 'nmf_owner', 'owner_share', 'n_motifs']]
member_tbl.to_csv(MEMBERS_CSV, index=False)

multi = member_tbl[member_tbl.token_n_concepts > 1]
print(f"wrote {os.path.abspath(MEMBERS_CSV)}: {len(member_tbl):,} concepts | {multi.token.nunique():,} group tokens ({len(multi):,} concepts) | "
      f"{int((~member_tbl.in_cohort).sum()):,} codeset members absent from the cohort")
top5 = {t for c in range(K) for t in H.iloc[c].nlargest(5).index}                           # group tokens that define a bundle, with their members
show = (multi[multi.token.isin(top5)].groupby('token', sort=False)
             .agg(owner=('nmf_owner', 'first'), members=('concept_id', 'size'), seen=('in_cohort', 'sum'), patients=('token_n_patients', 'first')))
print("\ngroup tokens among the top-5 loadings of a bundle:")
print(show.to_string())
member_tbl.head(8)

## 6 · Final table — top chest pain sequences and where they rank
Per NMF bundle: the highest-lift ordered patterns that contain a chest pain step, each with its rank among the bundle's scored top patterns, and under every top entity the median timing against the first chest pain. A post-filter of the scored candidates, so every score and verdict is the one computed above. Writes its own xlsx.

In [ ]:
COHORT = 'chestpain'
# ANCHOR-FOCUSED view of the final table (additional; the table above is untouched): per bundle, the top-TOPK_ANCHOR scored ordered patterns that contain a chest_pain step,
# with the position of that step (k of n). Same scores / verdicts as the table above (post-filter of hyb, no re-mining). Own xlsx file; Triage / literature / PMIDs left empty.
import re, itertools, collections
try:
    import openpyxl
except ImportError as e:
    raise ImportError("pip install openpyxl in this kernel's environment, then rerun this cell") from e
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter
from openpyxl.worksheet.datavalidation import DataValidation

TOPK, TOP_ENT = 3, 5            # patterns per bundle (highest lift first) / entities per bundle
RANK_BY = 'loading'             # entities ranked by NMF loading ('loading') or by share of the bundle's patients having the code ('prevalence')
import os; os.makedirs('outputs', exist_ok=True)
XLSX_FINAL = f"outputs/final_table_{COHORT}_k{K}_{VARIANT}.xlsx"
cdx = concept.drop_duplicates('concept_id').set_index('concept_id')
if 'member_tbl' not in globals():
    raise NameError("run the member-table cell (section 5, just above) first: the final table lists the members of its group tokens")
_grp = {t: g for t, g in member_tbl[member_tbl.token_n_concepts > 1].groupby('token', sort=False)}      # group tokens only, members by patients (desc)
TOP_MEM = 3                                                                  # members listed per group token; the full list is token_members_k{K}_{TAG}.csv

ANCHOR      = 'chest_pain'                                                    # cohort anchor group (codesets.yaml)
ANCHOR_NAME = ANCHOR.replace('_', ' ')
ANCHOR_TOK  = set(events.loc[eo.group_mask(events, ANCHOR), 'tag'])    # every concept-level token that belongs to the anchor group
TOPK_ANCHOR = 3                                                       # anchor patterns per bundle (highest lift first)

def has_anchor(pattern):
    return any(t in ANCHOR_TOK for t in pattern.split(' -> '))

_a0 = {}
def offset_text(c, members, t, min_n=20):
    """Median days from each patient's first {ANCHOR} step to their first occurrence of token t (negative = t came first), over the bundle's patients who have both."""
    if c not in _a0:                                                       # first anchor day per member, computed once per bundle
        _a0[c] = np.array([min((patseqs[i].dpos[x][0] for x in ANCHOR_TOK if x in patseqs[i].dpos), default=np.nan) for i in members])
    td = np.array([patseqs[i].dpos[t][0] if t in patseqs[i].dpos else np.nan for i in members])
    g  = (td - _a0[c])[~np.isnan(td - _a0[c])]
    if len(g) < min_n:
        return f"\n    vs first {ANCHOR_NAME}: too few patients with both (n={len(g)})"
    med = int(np.median(g))
    when = "same day as" if med == 0 else f"{abs(med):,} d {'before' if med < 0 else 'after'}"
    return f"\n    median {when} first {ANCHOR_NAME} ({(g < 0).mean():.0%} of them before)"

def _condense_name(n, limit=38):
    s = re.split(r' \(|;| This ', n)[0].strip()
    s = s if len(s) >= 8 else n                                      # packs like '{6 (azithromycin ...) } Pack' would be cut to '{6'
    return s if len(s) <= limit else s[:limit].rsplit(' ', 1)[0] + '…'

def label(token):
    m = re.search(r'__(\d+)$', token)
    if m and int(m.group(1)) in cdx.index:
        r = cdx.loc[int(m.group(1))]
        return f"{_condense_name(r.concept_name)} [{r.vocabulary_id} {r.concept_code}]"
    return f"{token.replace('_', ' ')} [code group]"

def pattern_text(pattern):
    parts = []
    for t, g in itertools.groupby(pattern.split(' -> ')):
        n = len(list(g)); parts.append(label(t) + (f" ×{n}" if n > 1 else ""))
    return ' → '.join(parts)

def members_text(tokens, top=TOP_MEM):
    """For every group token in `tokens` (a codeset tag standing for several concepts): count, then its most frequent members. '' when none is a group token."""
    out = []
    for t in dict.fromkeys(tokens):
        g = _grp.get(t)
        if g is None:
            continue
        seen = g[g.in_cohort]
        out.append(f"{t.replace('_', ' ')}: {len(g)} concepts ({len(seen)} seen in this cohort)")
        out += [f"  · {_condense_name(r.concept_name)} [{r.vocabulary_id} {r.concept_code}] (concept_id {r.concept_id}; {r.n_patients:,} patients)" for r in seen.head(top).itertuples()]
        if len(g) > top:
            out.append(f"  · +{len(g) - top} more")
    return "\n".join(out)

HEAD = ['Group (bundle)', 'n (%)', 'Most frequently OMOP clinical entities found', 'Rank (by lift)', 'Typical order of care when chest pain is present', "Position in the bundle's scored top patterns (by lift · by support)", 'Share of group with pattern', 'Share of everyone else with pattern', 'Held-out pass rate (family, 40 splits)', 'Holds up on held-out halves?', 'Times as common (lift)', 'Final verdict', 'Triage suggestion (Manual RUW / Routine / Unclear)', 'Order shown by dates?', 'Supported literature (Yes / Partly / None)', 'PMIDs', 'Members of the group tokens in the entities', 'Members of the group tokens in the pattern']
BUNDLE_COLS = (1, 2, 3, 13, 17)                   # merged down the rows of a bundle

table = []                                                          # list of (bundle number, row values)
for c in sorted(COMP):
    members = np.flatnonzero(comp == c)
    cnt   = collections.Counter(t for i in members for t in set(db[i]))
    share = lambda t: cnt.get(t, 0) / max(len(members), 1)
    score = H.iloc[c] if RANK_BY == 'loading' else pd.Series({t: share(t) for t in H.columns})
    ents  = "\n".join(f"{i}. {label(t)} ({share(t):.0%} of bundle){offset_text(c, members, t)}" for i, t in enumerate(score.nlargest(TOP_ENT).index, 1))
    head  = [f"c{c} · {COMP[c][1]}", f"{len(members):,} ({len(members) / len(comp):.1%})", ents]
    ent_mem = members_text(score.nlargest(TOP_ENT).index)                           # members of the group tokens among the entities (bundle-level, merged)
    pool  = hyb.get(c)
    pool  = pool if pool is not None and len(pool) else None
    if pool is not None:
        pool = pool.assign(rk_lift=np.arange(1, len(pool) + 1), rk_sup=pool.sup_in.rank(ascending=False, method='min').astype(int))   # hyb[c] is lift-sorted
        has  = pool.pattern.map(has_anchor)
    d     = pool[has].head(TOPK_ANCHOR) if pool is not None else None
    head[1] += f"\n{int(has.sum()) if pool is not None else 0} of {len(pool) if pool is not None else 0} scored patterns contain {ANCHOR_NAME}"
    if d is None or not len(d):
        table.append((c, head + ['', f"no scored pattern contains {ANCHOR_NAME} (bundle has {len(pool) if pool is not None else 0} scored patterns)"] + [''] * (len(HEAD) - 7) + [ent_mem, '']))
        continue
    for rk, (_, r) in enumerate(d.iterrows(), 1):
        ok = r.fam_holdout_rate_lo >= HO_MIN_RATE_LO
        verdict = 'Unstable' if not ok else ('Suggestive' if r.same_day_bundle else 'Solid')
        row = (head if rk == 1 else ['', '', '']) + [rk, pattern_text(r.pattern), f"{int(r.rk_lift)} of {len(pool)} by lift · {int(r.rk_sup)} of {len(pool)} by support", float(r.sup_in), float(r.sup_out), float(r.fam_holdout_rate), 'Yes' if ok else 'No',
                                                     float(r.lift), verdict, '', 'Same-day bundle' if r.same_day_bundle else 'Yes', '', '',
                                                     ent_mem if rk == 1 else '', members_text(r.pattern.split(' -> '))]
        table.append((c, row))

FILL = {'Yes': 'D6F0DE', 'Partly': 'FBEFC9', 'No': 'F8D7D7', 'Solid': 'D6F0DE', 'Suggestive': 'FBEFC9', 'Unstable': 'F8D7D7', 'Same-day bundle': 'FBEFC9'}
F = lambda **kw: Font(name='Arial', size=10, **kw)
thin = Side(style='thin', color='D0D5DB')
wb = openpyxl.Workbook(); ws = wb.active; ws.title = f'{COHORT} k={K}'
ws.append(HEAD)
for j in range(1, len(HEAD) + 1):
    h = ws.cell(1, j); h.font = Font(name='Arial', size=10, bold=True, color='FFFFFF')
    h.fill = PatternFill('solid', fgColor='334E68'); h.alignment = Alignment(wrap_text=True, vertical='center', horizontal='center')
ws.row_dimensions[1].height = 44
for _, row in table:
    ws.append(row)
dv = DataValidation(type='list', formula1='"Yes,Partly,None"', allow_blank=True); ws.add_data_validation(dv)
LEFT = {1, 3, 5, 13, 17, 18}
for i in range(2, len(table) + 2):
    ws.row_dimensions[i].height = 64
    for j in range(1, len(HEAD) + 1):
        c_ = ws.cell(i, j); c_.font = F(bold=(j == 1)); c_.border = Border(bottom=thin)
        c_.alignment = Alignment(wrap_text=True, vertical='top', horizontal='left' if j in LEFT else 'center')
    for j in (7, 8, 9): ws.cell(i, j).number_format = '0.0%'
    ws.cell(i, 11).number_format = '0.0'
    for j in (10, 12, 14):
        v = ws.cell(i, j).value
        if v in FILL: ws.cell(i, j).fill = PatternFill('solid', fgColor=FILL[v]); ws.cell(i, j).font = F(bold=True)
    dv.add(ws.cell(i, 15))
r0 = 2
for c, grp in itertools.groupby([b for b, _ in table]):               # one merged block per bundle
    n = len(list(grp)); r1 = r0 + n - 1
    if n > 1:
        for j in BUNDLE_COLS: ws.merge_cells(start_row=r0, end_row=r1, start_column=j, end_column=j)
    lines = lambda v: str(v or '').count('\n') + 1                      # rows grow so the members text (13 pt per line) is not cut off; Excel caps a row at 409
    for rr in range(r0, r1 + 1):
        ws.row_dimensions[rr].height = min(409, max(ws.row_dimensions[rr].height, 13 * lines(ws.cell(r0, 17).value) / n, 13 * lines(ws.cell(rr, 18).value)))
    r0 = r1 + 1
for j, w in enumerate([30, 22, 70, 8, 80, 22, 12, 13, 11, 11, 11, 12, 34, 14, 16, 16, 70, 70], 1):
    ws.column_dimensions[get_column_letter(j)].width = w
ws.freeze_panes = 'B2'
notes = [
    "Notes",
    f"Source: fused Day 1 + Day 3 notebook, chestpain run ({VARIANT}{', +/- ' + str(WINDOW_MONTHS) + ' months around any chest pain event' if WINDOW_MONTHS else ''}), NMF k={K} on concept-level tokens, {len(db):,} patients. Bundles are the NMF components in order c0, c1, ...; n (%) = patients whose dominant component is the bundle.",
    f"Entities: the top {TOP_ENT} OMOP concepts of the bundle ranked by {'NMF loading' if RANK_BY == 'loading' else 'share of the bundle patients having the code'}; the % is the share of the bundle's patients with that code.",
    f"Patterns: the {TOPK_ANCHOR} highest-lift scored ordered patterns of each bundle that contain a {ANCHOR_NAME} step (any {ANCHOR_NAME} concept), from the same candidates as the main table (ordered patterns of 3+ steps, each present in >={REL_SUP:.0%} of the bundle). The second line of each n (%) cell says how many of the bundle's scored patterns contain {ANCHOR_NAME}.",
    f"Entities: under each entity, the median number of days from the patient's first {ANCHOR_NAME} step to their FIRST occurrence of that entity (before / after), over the bundle patients who have both, and the share of them for whom it came first. Repeated entities (labs, drugs) are read at their first occurrence.",
    "Position among the bundle's scored top patterns: each bundle's candidates are its 100 most frequent ordered patterns (3+ steps) that pass the support floor, scored for lift; the pattern's place in that list by lift (1 = most specific to the bundle) and by support (1 = most frequent), out of the N patterns scored. A bundle with no such pattern is shown as such, not filled.",
    "Times as common = (share inside bundle + 0.01) / (share outside + 0.01). Outside = random sample of 1,500 patients from other bundles.",
    f"Held-out pass rate: {len(HO_SEEDS)} random 50/50 splits of the whole bundle x 2 directions = {2 * len(HO_SEEDS)} runs; a run passes if, on the half not used, support is >= {REL_SUP:.0%} and the lower 95% bound of the odds ratio vs all other patients is >= {HO_MIN_OR:g}. Near-duplicate patterns are one family (a run passes if any member does). Holds = family lower 95% bound >= {HO_MIN_RATE_LO:.0%}.",
    "Final verdict (chest pain has no bootstrap / re-test run, so the held-out result is used): Solid = holds and order shown by dates; Suggestive = holds but a same-day bundle; Unstable = does not hold.",
    f"Order shown by dates? 'Same-day bundle' = with every step required on a strictly later day, support falls below {SAME_DAY_RATIO:.0%} of its table value: the steps mostly occur together on one day, so the order is a tie-break, not evidence.",
    "Triage suggestion, Supported literature (Yes / Partly / None, drop-down) and PMIDs are left empty for the review.",
    "'×N' = the same concept appears N times in a row. '[code group]' = a curated group of several codes.",
    f"Members columns: for every group token (a codeset tag standing for several concepts) in the entities / in the pattern, how many concepts it holds, how many occur in this cohort, and its {TOP_MEM} most frequent members with concept_id and patient count. Group tokens are listed once; full list: token_members_k{K}_{TAG}.csv. Empty = no group token (all concept-level).",
]
for k_, t in enumerate(notes):
    ws.cell(len(table) + 3 + k_, 1, t).font = F(bold=(k_ == 0))
wb.save(XLSX_FINAL)
print(f"wrote {os.path.abspath(XLSX_FINAL)}  ({len(COMP)} bundles, {len(table)} rows)")
pd.DataFrame([r for _, r in table], columns=HEAD).iloc[:, :12]


## 7 · Animation export — the trajectory graph as a self-contained HTML
Writes `trajectory_k{K}_{TAG}.json` (one median and SD per entity, the edges, the lane layout of the time-axis graph above; no patient-level data) and builds `trajectory_k{K}_{TAG}_animation.html` with `trajectory_anim.py` (d3 inlined, no network). Needs the graph cells above in the same kernel: `G` (cell 58) and `_first`, `pos_t`, `lane_of`, … (cell 60). In the page, each entity gets about 60 dots drawn from Normal(median, SD) of the days from the first chest pain; a time cursor sweeps and lights the dots it passes.

In [ ]:
# Animation export: the numbers behind the trajectory graph (cells 58 and 60) as a small JSON, then one self-contained HTML with the animation.
# Reads from this kernel: G (cell 58); _first, lane_of, LANES, LANE_COL, centers, heights, pos_t, tf, tick_d, MIN_N, _have (cell 60).
# One median / SD per entity: no patient-level data is written. The page draws about 60 dots per entity from Normal(median, SD) of the days
# from the first chest pain to the entity's first occurrence; the footer of the HTML says what that does and does not mean.
import json, importlib
import trajectory_anim
importlib.reload(trajectory_anim)
assert abs(tf(30) - np.log(2)) < 1e-12, "cell 60 changed the time compression: update tf_days here and in the page"
JSON_TRAJ = f"trajectory_k{K}_{TAG}.json"

nodes_j = []
for g in G.nodes():
    v = np.asarray(_first[g], dtype=float)
    assert len(v) >= MIN_N, f"{g}: only {len(v)} patients with timing; cell 60 parks it at the right edge"
    q25, q75 = np.percentile(v, [25, 75])
    nodes_j.append(dict(id=g, lane=lane_of[g], n=int(len(v)), median=float(np.median(v)), mean=float(v.mean()), sd=float(v.std(ddof=1)),
                        q25=float(q25), q75=float(q75), p_before=float((v < 0).mean()), x=float(pos_t[g][0]), y=float(pos_t[g][1])))
traj = dict(title=f"{ANCHOR_NAME.capitalize()} trajectory graph, {VARIANT} (codesets {CODESETS_FILE}), k={K}", n_patients=int(len(_have)), k=int(K), tag=TAG, anchor=ANCHOR,
            tf_days=30, ticks=[int(d) for d in tick_d],
            lanes=[dict(name=l, color=LANE_COL.get(l, '#e2e8f0'), y=float(centers[l]), h=float(heights[l])) for l in LANES],
            nodes=nodes_j,
            edges=[dict(source=a, target=b, weight=int(w['weight'])) for a, b, w in sorted(G.edges(data=True), key=lambda e: -e[2]['weight'])])
with open(JSON_TRAJ, 'w', encoding='utf-8') as f:
    json.dump(traj, f, indent=1, ensure_ascii=False)
HTML_TRAJ = trajectory_anim.build(JSON_TRAJ)
print(f"wrote {os.path.abspath(JSON_TRAJ)} ({os.path.getsize(JSON_TRAJ) / 1024:.1f} KB) and {HTML_TRAJ} ({os.path.getsize(HTML_TRAJ) / 1024:,.0f} KB)")
pd.DataFrame(nodes_j).drop(columns=['x', 'y']).round(2).sort_values('median')

## 8 · Ladder: plain codes, crossed only, plain + crossed, plain + sex and age band, plain + risk flags

The same NMF settings (`NMF_KW`, k = the k fitted above) on five versions of the matrix, each fitted on the same training patients and scored on the same held-out patients:

- **A plain codes**: the AHA k=8 matrix (the reference);
- **S crossed only**: only the code x sex x age band (at the event) columns, the matrix fitted above when `AGE_SEX_CROSS = 'crossed'`. Sex does not change, so a woman and a man share no column: read the NMI column first;
- **plain + S**: the plain codes and the crossed columns together (the crossed columns weighted by `W_S`);
- **plain + D**: the plain codes and the patient's sex and age band at each chest pain event, one-hot (the `'extended'` matrix, weighted by `W_D`);
- **plain + R**: the plain codes and Sean's two risk flags (man aged 45+, woman aged 55+ at any chest pain visit), one-hot (the `'explicit'` matrix, weighted by `W_R`).

Read it as a comparison of what the columns cost and buy, not as a ranking:

- **held-out error** is the relative reconstruction error on unseen patients (it is not comparable across rows with very different column counts; read it with the stability columns);
- **stability** is the bootstrap refit cosine and ARI used in the k sweep (0.90 / 0.75 as in the cells above);
- **NMI with age-sex stratum** (at the first chest pain) says how much the components are age-sex strata;
- **ARI vs plain codes (A)** and **vs AHA k=8** say how far the patients' components moved.

With `AGE_SEX_CROSS = 'plain'` the crossed bag is not built, so the two crossed rows do not run (the plain and plain + D rows do).

*In plain terms: five runs: the original columns, only the age-sex split columns, both, the original columns with the patient's sex and age band added, and the original columns with the two risk flags added. If a run's bundles stop being reproducible, or turn into age-sex groups, that is visible here.*

In [ ]:
LADDER = {'A plain codes': ([], True), 'S crossed only': (['S'], False), 'plain + S': (['S'], True), 'plain + D': (['D'], True), 'plain + R': (['R'], True)}   # name: (blocks, keep the plain columns)
LADDER = {n: v for n, v in LADDER.items() if all(x in BLOCKS for x in v[0])}
K_L    = nmf.n_components_
perm_l = np.random.default_rng(0).permutation(len(person_ids))
N_TR_L = min(20_000, int(0.6 * len(perm_l)))
tr_l   = perm_l[:N_TR_L]
ho_l   = perm_l[N_TR_L:N_TR_L + min(10_000, len(perm_l) - N_TR_L)]
strat_l = demo.reindex(person_ids)['stratum']
ok_l    = strat_l.notna().to_numpy()
_z   = np.load(BASE_NPZ) if os.path.exists(BASE_NPZ) else None
base_l = pd.Series(_z['labels'], index=_z['pids']) if _z is not None else None

labs_l, rows_l = {}, []
for name, (blocks, plain) in LADDER.items():
    XL, cols_l = assemble(blocks, plain)
    t0   = time.perf_counter()
    m_l  = NMF(n_components=K_L, **NMF_KW).fit(XL[tr_l])
    err  = rel_err(XL[ho_l], m_l.transform(XL[ho_l]), m_l.components_)
    cos_l, ari_l, _ = nmf_stability(m_l, XL[tr_l], n_boot=N_BOOT, n_sample=len(tr_l), n_eval=len(tr_l), seed=1)
    lab_l = m_l.transform(XL).argmax(1)
    labs_l[name] = lab_l
    share_l = np.bincount(lab_l, minlength=K_L) / len(lab_l)
    row = {'matrix': name, 'columns': XL.shape[1], 'held-out error': round(err, 3),
           'stability cos': round(float(cos_l.mean()), 3), 'stability ARI': round(float(ari_l.mean()), 3),
           'unstable components': int((cos_l.mean(0) < .75).sum()),
           'NMI with age-sex stratum': round(normalized_mutual_info_score(strat_l[ok_l], lab_l[ok_l]), 3),
           'components >= 1%': int((share_l >= .01).sum()), 'largest component %': round(100 * share_l.max(), 1)}
    first = next(iter(labs_l.values()))
    row['ARI vs plain codes (A)'] = round(adjusted_rand_score(first, lab_l), 3)
    if base_l is not None:
        both_l = base_l.index.intersection(pd.Index(person_ids))
        row['ARI vs AHA k=8'] = round(adjusted_rand_score(base_l.loc[both_l], pd.Series(lab_l, index=person_ids).loc[both_l]), 3)
    row['seconds'] = round(time.perf_counter() - t0)
    rows_l.append(row)
    print(f"{name}: done in {row['seconds']} s")
ladder_tbl = pd.DataFrame(rows_l).set_index('matrix')
display(ladder_tbl)